<a href="https://colab.research.google.com/github/4bdurr/RDHEI_in_medical_volumes/blob/main/KBJ_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile rdh_encrypted_3d_medical_volumes.py
"""
Residual-Distribution-Aware Reversible Data Hiding (RDH) in Encrypted 3D Medical Volumes
==========================================================================================
Stage 0 + Stage 1 Experimental Platform
"""

from __future__ import annotations
from dataclasses import dataclass, field
import glob
import heapq
import os
import struct
import tarfile
import time
from typing import Dict, List, Optional, Tuple

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import numpy as np

try:
  from numba import njit
  HAVE_NUMBA = True
except ImportError:
  HAVE_NUMBA = False

  def njit(*args, **kwargs):
    def wrap(fn):
      return fn
    if len(args) == 1 and callable(args[0]) and not kwargs:
      return args[0]
    return wrap

try:
  import nibabel as nib
  HAVE_NIBABEL = True
except ImportError:
  HAVE_NIBABEL = False

B_WIDTH = 32
SYMBOL_BITS = 7
HEADER_BITS = 512
INT64_MAX = (1 << 63) - 1

PREDICTOR_FAMILY_IDS = {
    "Causal13": 0,
    "MED": 1,
    "WCP_PAPER": 2,
    "WCP": 2,
    "WCP_FAIR": 2,
    "WCP_INSLICE": 2,
    "AWCP_V1": 3,
    "AWCP": 3,
    "AWCP_V1_INT": 3,
    "Causal13_FLOAT": 0,
    "WCP_PAPER_FLOAT": 2,
}

VARIANT_IDS = {
    "Causal13": 0,
    "MED": 1,
    "WCP_PAPER": 2,
    "WCP_FAIR": 3,
    "WCP_INSLICE": 4,
    "AWCP_V1": 5,
    "Causal13_FLOAT": 6,
    "WCP_PAPER_FLOAT": 7,
    "AWCP_V1_INT": 8,
}
VARIANT_IDS["WCP"] = VARIANT_IDS["WCP_PAPER"]
VARIANT_IDS["AWCP"] = VARIANT_IDS["AWCP_V1"]

ID_TO_VARIANT = {v: k for k, v in VARIANT_IDS.items() if k not in ("WCP", "AWCP")}

EPS_SPEC_TABLE = (
    (0.0, 0, 1),
    (1e-8, 1, 100000000),
    (1e-6, 1, 1000000),
    (1e-5, 1, 100000),
    (1e-4, 1, 10000),
    (1e-3, 1, 1000),
    (1e-2, 1, 100),
    (1e-1, 1, 10),
    (1.0, 1, 1),
    (10.0, 10, 1),
    (100.0, 100, 1),
    (1000.0, 1000, 1),
)
EPS_TABLE = tuple(item[0] for item in EPS_SPEC_TABLE)


def eps_to_code(eps: float) -> int:
  diffs = [abs(eps - val) for val in EPS_TABLE]
  best_idx = int(np.argmin(diffs))
  if diffs[best_idx] > 1e-12 * max(1.0, abs(eps)):
    raise ValueError(f"eps={eps} is not in EPS_TABLE={EPS_TABLE}")
  return best_idx


def code_to_eps(code: int) -> float:
  if not (0 <= code < len(EPS_SPEC_TABLE)):
    raise ValueError(f"Invalid eps_code={code}")
  return EPS_SPEC_TABLE[code][0]


def eps_to_rational(eps: float) -> Tuple[int, int]:
  code = eps_to_code(eps)
  _, num, den = EPS_SPEC_TABLE[code]
  return int(num), int(den)


def ensure_tar_extracted(
    data_dir: str = "Task05_Prostate",
    search_dirs: Tuple[str, ...] = (".", "/content", "/content/drive/MyDrive"),
) -> str:
  img_tr = os.path.join(data_dir, "imagesTr")
  if os.path.isdir(img_tr):
    return data_dir

  content_alt = os.path.join("/content", data_dir)
  if os.path.isdir(os.path.join(content_alt, "imagesTr")):
    return content_alt

  tar_name = f"{os.path.basename(data_dir.rstrip('/'))}.tar"
  for sdir in search_dirs:
    candidate_tar = os.path.join(sdir, tar_name)
    if os.path.isfile(candidate_tar):
      print(f"[AUTO-EXTRACT] Mengekstrak '{candidate_tar}' ke direktori kerja...")
      with tarfile.open(candidate_tar, "r:*") as tf:
        tf.extractall(path=".")
      if os.path.isdir(img_tr):
        return data_dir
      if os.path.isdir(os.path.join(content_alt, "imagesTr")):
        return content_alt
  return data_dir


def list_valid_nifti_files(data_dir: str, subdir: str = "imagesTr") -> List[str]:
  resolved_dir = ensure_tar_extracted(data_dir)
  pattern = os.path.join(resolved_dir, subdir, "*.nii*")
  raw_files = sorted(glob.glob(pattern))
  return [
      f for f in raw_files
      if not os.path.basename(f).startswith(".")
      and not os.path.basename(f).startswith("._")
  ]


def effective_bit_length(mag: int) -> int:
  return int(abs(int(mag))).bit_length()


@njit(cache=True)
def _compute_bit_length_flat_numba(mag_flat: np.ndarray) -> np.ndarray:
  n = mag_flat.shape[0]
  out = np.zeros(n, dtype=np.int64)
  for idx in range(n):
    v = mag_flat[idx]
    if v < 0:
      v = -v
    bl = 0
    while v > 0:
      v >>= 1
      bl += 1
    out[idx] = bl
  return out


def compute_bit_length_int64(mag: np.ndarray) -> np.ndarray:
  mag_i64 = np.abs(np.asarray(mag, dtype=np.int64))
  if HAVE_NUMBA:
    flat = _compute_bit_length_flat_numba(mag_i64.reshape(-1))
    return flat.reshape(mag_i64.shape)
  x = mag_i64.astype(np.uint64).copy()
  ell = np.zeros(x.shape, dtype=np.int64)
  for shift in (32, 16, 8, 4, 2, 1):
    mask = x >= (np.uint64(1) << np.uint64(shift))
    ell += mask.astype(np.int64) * shift
    x = np.where(mask, x >> np.uint64(shift), x)
  ell += (x > 0).astype(np.int64)
  return ell


def preprocess_volume(vol: np.ndarray) -> Tuple[np.ndarray, float, int]:
  vol = np.asarray(vol, dtype=np.float32)
  if not np.all(np.isfinite(vol)):
    raise ValueError("Volume contains non-finite samples; out of scope of the framework.")
  if not np.array_equal(vol, np.round(vol)):
    raise ValueError("Volume contains non-integer samples; out of scope of the framework.")
  vmin = float(vol.min())
  delta = min(vmin, 0.0)
  vprime = (vol.astype(np.float64) - delta).astype(np.int64)
  M = int(vprime.max())
  be = max(1, M.bit_length()) if M > 0 else 1
  return vprime, delta, be


@dataclass
class NIfTITransformMeta:
  original_shape: Tuple[int, int, int]
  canonical_applied: bool
  swap_inplane_applied: bool
  ornt_transform: Optional[np.ndarray] = None
  inv_ornt_transform: Optional[np.ndarray] = None
  original_affine: Optional[np.ndarray] = None


def invert_nifti_transform(vol: np.ndarray, meta: NIfTITransformMeta) -> np.ndarray:
  out = np.asarray(vol)
  if meta.swap_inplane_applied:
    out = np.swapaxes(out, 0, 1)
  if meta.canonical_applied and meta.inv_ornt_transform is not None:
    if not HAVE_NIBABEL:
      raise ImportError("nibabel is required to invert orientation transforms.")
    out = nib.orientations.apply_orientation(out, meta.inv_ornt_transform)
  return out


def load_nifti_volume(
    path: str,
    component: int = 0,
    roi: Optional[Tuple[Tuple[int, int], Tuple[int, int], Tuple[int, int]]] = None,
    canonical: bool = False,
    swap_inplane_axes: bool = False,
    return_meta: bool = False,
):
  if not HAVE_NIBABEL:
    raise ImportError("nibabel is required to load NIfTI files: pip install nibabel")

  img = nib.load(path)
  orig_affine = img.affine.copy()
  raw_all = np.asanyarray(img.dataobj)

  if raw_all.ndim == 4:
    if not (0 <= component < raw_all.shape[-1]):
      raise ValueError(f"component={component} out of range for shape {raw_all.shape}")
    raw_3d = np.asarray(raw_all[..., component])
  elif raw_all.ndim == 3:
    raw_3d = np.asarray(raw_all)
  else:
    raise ValueError(f"Unsupported NIfTI rank: {raw_all.ndim} (expected 3 or 4)")

  ornt_trans = None
  inv_ornt_trans = None

  if canonical:
    orig_ornt = nib.io_orientation(orig_affine)
    canon_img = nib.as_closest_canonical(img)
    canon_ornt = nib.io_orientation(canon_img.affine)
    ornt_trans = nib.orientations.ornt_transform(orig_ornt, canon_ornt)
    inv_ornt_trans = nib.orientations.ornt_transform(canon_ornt, orig_ornt)

    assert int(ornt_trans[2, 0]) == 2, (
        f"Canonical reorientation permuted the 3rd (slice) axis! ornt_transform={ornt_trans.tolist()}"
    )
    data_3d = nib.orientations.apply_orientation(raw_3d, ornt_trans)
    reconstructed_raw = nib.orientations.apply_orientation(data_3d, inv_ornt_trans)
    assert np.array_equal(reconstructed_raw, raw_3d), "Canonical transform is not exactly invertible!"
  else:
    data_3d = raw_3d.copy()

  if swap_inplane_axes:
    data_3d = np.swapaxes(data_3d, 0, 1)

  meta = NIfTITransformMeta(
      original_shape=tuple(int(x) for x in raw_3d.shape),
      canonical_applied=canonical,
      swap_inplane_applied=swap_inplane_axes,
      ornt_transform=ornt_trans,
      inv_ornt_transform=inv_ornt_trans,
      original_affine=orig_affine,
  )

  if canonical or swap_inplane_axes:
    check_back = invert_nifti_transform(data_3d, meta)
    assert np.array_equal(check_back, raw_3d), "Combined NIfTI transform failed invertibility check!"

  vol = np.asarray(data_3d, dtype=np.float64)
  vol_rounded = np.round(vol).astype(np.float32)
  if not np.array_equal(vol, vol_rounded.astype(np.float64)):
    n_bad = int(np.sum(vol != vol_rounded.astype(np.float64)))
    raise ValueError(f"{n_bad} voxel(s) are non-integer; out of the framework's scope.")

  if roi is not None:
    (i0, i1), (j0, j1), (k0, k1) = roi
    vol_rounded = vol_rounded[i0:i1, j0:j1, k0:k1]

  if return_meta:
    return vol_rounded, meta
  return vol_rounded


def _flatten_kij(arr3d: np.ndarray) -> np.ndarray:
  return np.transpose(arr3d, (2, 0, 1)).reshape(-1)


def _unflatten_kij(flat: np.ndarray, shape: Tuple[int, int, int]) -> np.ndarray:
  W, H, D = shape
  return np.transpose(flat.reshape(D, W, H), (1, 2, 0))


def check_awcp_int64_overflow_bound(
    be: int, M: int, wz_max: int, wxy: int, eps_num: int, eps_den: int, var_win: int = 2
) -> dict:
  M_val = max(int(M), (1 << int(be)) - 1, 1)
  c_xy_max = 4
  c_z_max = 9
  m_v_max = 4 if int(var_win) == 4 else 2
  delta_max = c_xy_max * M_val
  V_num_max = m_v_max * (delta_max * delta_max)
  N_tau_max = int(eps_den) * V_num_max + int(eps_num) * m_v_max * (c_xy_max * c_xy_max)
  D_tau_max = N_tau_max + int(eps_den) * m_v_max * (delta_max * delta_max)

  S_z_max = c_z_max * M_val
  S_xy_max = c_xy_max * M_val
  Num_max = (
      int(wz_max) * N_tau_max * S_z_max * c_xy_max
      + int(wxy) * D_tau_max * S_xy_max * c_z_max
  )
  Den_max = (int(wz_max) * N_tau_max + int(wxy) * D_tau_max) * c_z_max * c_xy_max
  worst_case_val = 2 * Num_max + Den_max
  fits_int64 = worst_case_val <= INT64_MAX
  return dict(
      be=be,
      M=M,
      worst_case_bits=int(worst_case_val).bit_length(),
      fits_unconditional_int64=fits_int64,
      worst_case_val=worst_case_val,
  )


@njit(cache=True)
def _gcd_int64(a: int, b: int) -> int:
  while b != 0:
    a, b = b, a % b
  return a if a > 0 else 1


@njit(cache=True)
def _u128_mul_u64(a: np.uint64, b: np.uint64) -> Tuple[np.uint64, np.uint64]:
  mask32 = np.uint64(0xFFFFFFFF)
  a_lo = a & mask32
  a_hi = a >> np.uint64(32)
  b_lo = b & mask32
  b_hi = b >> np.uint64(32)

  p0 = a_lo * b_lo
  p1 = a_lo * b_hi
  p2 = a_hi * b_lo
  p3 = a_hi * b_hi

  cy = ((p0 >> np.uint64(32)) + (p1 & mask32) + (p2 & mask32)) >> np.uint64(32)
  lo = p0 + ((p1 & mask32) << np.uint64(32)) + ((p2 & mask32) << np.uint64(32))
  hi = p3 + (p1 >> np.uint64(32)) + (p2 >> np.uint64(32)) + cy
  return hi, lo


@njit(cache=True)
def _u128_add(
    a_hi: np.uint64, a_lo: np.uint64, b_hi: np.uint64, b_lo: np.uint64
) -> Tuple[np.uint64, np.uint64]:
  lo = a_lo + b_lo
  cy = np.uint64(1) if lo < a_lo else np.uint64(0)
  hi = a_hi + b_hi + cy
  return hi, lo


@njit(cache=True)
def _u128_mul_small(
    a_hi: np.uint64, a_lo: np.uint64, k: np.uint64
) -> Tuple[np.uint64, np.uint64]:
  p_hi, p_lo = _u128_mul_u64(a_lo, k)
  _, high_lo = _u128_mul_u64(a_hi, k)
  return p_hi + high_lo, p_lo


@njit(cache=True)
def _u128_le(
    a_hi: np.uint64, a_lo: np.uint64, b_hi: np.uint64, b_lo: np.uint64
) -> bool:
  if a_hi != b_hi:
    return a_hi < b_hi
  return a_lo <= b_lo


@njit(cache=True)
def _awcp_v1_int_fallback_u128(
    N_tau: int,
    D_tau: int,
    wz_max: int,
    wxy: int,
    s_inter: int,
    c_inter: int,
    s_intra: int,
    c_intra: int,
) -> int:
  uN = np.uint64(N_tau)
  uD = np.uint64(D_tau)
  u_wz_coeff = np.uint64(wz_max * s_inter * c_intra)
  u_wxy_coeff = np.uint64(wxy * s_intra * c_inter)

  num1_hi, num1_lo = _u128_mul_u64(uN, u_wz_coeff)
  num2_hi, num2_lo = _u128_mul_u64(uD, u_wxy_coeff)
  num_hi, num_lo = _u128_add(num1_hi, num1_lo, num2_hi, num2_lo)

  den1_hi, den1_lo = _u128_mul_u64(uN, np.uint64(wz_max * c_inter * c_intra))
  den2_hi, den2_lo = _u128_mul_u64(uD, np.uint64(wxy * c_inter * c_intra))
  den_hi, den_lo = _u128_add(den1_hi, den1_lo, den2_hi, den2_lo)

  t_hi, t_lo = _u128_add(num_hi, num_lo, num_hi, num_lo)
  t_hi, t_lo = _u128_add(t_hi, t_lo, den_hi, den_lo)

  s_hi, s_lo = _u128_add(den_hi, den_lo, den_hi, den_lo)

  mu_z_floor = s_inter // c_inter
  mu_z_ceil = (s_inter + c_inter - 1) // c_inter
  mu_xy_floor = s_intra // c_intra if c_intra > 0 else 0
  mu_xy_ceil = (s_intra + c_intra - 1) // c_intra if c_intra > 0 else 0

  lo_q = min(mu_z_floor, mu_xy_floor)
  hi_q = max(mu_z_ceil, mu_xy_ceil)

  while lo_q < hi_q:
    mid = (lo_q + hi_q + 1) >> 1
    p_hi, p_lo = _u128_mul_small(s_hi, s_lo, np.uint64(mid))
    if _u128_le(p_hi, p_lo, t_hi, t_lo):
      lo_q = mid
    else:
      hi_q = mid - 1
  return int(lo_q)


@njit(cache=True)
def _round_half_up_nonneg(num: int, den: int) -> int:
  return (2 * num + den) // (2 * den)


@njit(cache=True)
def _predict_single_voxel(
    src: np.ndarray,
    i: int,
    j: int,
    k: int,
    W: int,
    H: int,
    D: int,
    variant_id: int,
    wz_max: int,
    wxy: int,
    eps: float,
    eps_num: int,
    eps_den: int,
    var_win: int,
) -> int:
  if variant_id == 1:
    A = src[i, j - 1, k] if j - 1 >= 0 else 0
    Bv = src[i - 1, j, k] if i - 1 >= 0 else 0
    C = src[i - 1, j - 1, k] if (i - 1 >= 0 and j - 1 >= 0) else 0
    Dv = src[i, j, k - 1] if k - 1 >= 0 else 0
    L = A + Bv + Dv - C
    mmin = min(A, min(Bv, Dv))
    mmax = max(A, max(Bv, Dv))
    if L >= mmax:
      return mmin
    elif L <= mmin:
      return mmax
    else:
      return L

  s_intra = 0
  c_intra = 0
  val_j_m1 = 0
  val_i_m1 = 0
  val_diag_ul = 0
  val_diag_ur = 0
  has_j_m1 = False
  has_i_m1 = False
  has_diag_ul = False
  has_diag_ur = False

  if j - 1 >= 0:
    val_j_m1 = src[i, j - 1, k]
    s_intra += val_j_m1
    c_intra += 1
    has_j_m1 = True
  if i - 1 >= 0:
    val_i_m1 = src[i - 1, j, k]
    s_intra += val_i_m1
    c_intra += 1
    has_i_m1 = True
    if j - 1 >= 0:
      val_diag_ul = src[i - 1, j - 1, k]
      s_intra += val_diag_ul
      c_intra += 1
      has_diag_ul = True
    if j + 1 < H:
      val_diag_ur = src[i - 1, j + 1, k]
      s_intra += val_diag_ur
      c_intra += 1
      has_diag_ur = True

  s_inter = 0
  c_inter = 0
  if k > 0:
    for di in range(-1, 2):
      ii = i + di
      if ii < 0 or ii >= W:
        continue
      for dj in range(-1, 2):
        jj = j + dj
        if jj < 0 or jj >= H:
          continue
        s_inter += src[ii, jj, k - 1]
        c_inter += 1

  if variant_id == 0:
    total_s = s_inter + s_intra
    total_c = c_inter + c_intra
    if total_c == 0:
      return 0
    return _round_half_up_nonneg(total_s, total_c)

  if variant_id == 6:
    total_s = s_inter + s_intra
    total_c = c_inter + c_intra
    if total_c == 0:
      return 0
    return int(round(float(total_s) / float(total_c)))

  if variant_id == 4:
    if c_intra == 0:
      return 0
    return _round_half_up_nonneg(s_intra, c_intra)

  if variant_id == 2 or variant_id == 3:
    wz = wz_max
    if wz + wxy == 0:
      return 0
    if c_inter == 0:
      if c_intra == 0:
        return 0
      if variant_id == 3:
        return _round_half_up_nonneg(s_intra, c_intra)
      else:
        num = wxy * s_intra
        den = (wz + wxy) * c_intra
        return _round_half_up_nonneg(num, den)
    else:
      if c_intra == 0:
        num = wz * s_inter
        den = (wz + wxy) * c_inter
        return _round_half_up_nonneg(num, den)
      num = wz * s_inter * c_intra + wxy * s_intra * c_inter
      den = (wz + wxy) * c_inter * c_intra
      return _round_half_up_nonneg(num, den)

  if variant_id == 7:
    mu_z = (float(s_inter) / float(c_inter)) if c_inter > 0 else 0.0
    mu_xy = (float(s_intra) / float(c_intra)) if c_intra > 0 else 0.0
    return int(round((float(wz_max) * mu_z + float(wxy) * mu_xy) / float(wz_max + wxy)))

  if variant_id == 5:
    mu_xy = (float(s_intra) / float(c_intra)) if c_intra > 0 else 0.0
    if k == 0:
      return int(round(mu_xy))
    mu_z = (float(s_inter) / float(c_inter)) if c_inter > 0 else 0.0
    v_prev = float(src[i, j, k - 1])

    var_xy = 0.0
    if var_win == 4:
      if c_intra > 0:
        acc_var = 0.0
        if has_j_m1:
          d0 = float(val_j_m1) - mu_xy
          acc_var += d0 * d0
        if has_i_m1:
          d1 = float(val_i_m1) - mu_xy
          acc_var += d1 * d1
        if has_diag_ul:
          d2 = float(val_diag_ul) - mu_xy
          acc_var += d2 * d2
        if has_diag_ur:
          d3 = float(val_diag_ur) - mu_xy
          acc_var += d3 * d3
        var_xy = acc_var / float(c_intra)
    else:
      if has_j_m1 and has_i_m1:
        d0 = float(val_j_m1) - mu_xy
        d1 = float(val_i_m1) - mu_xy
        var_xy = 0.5 * (d0 * d0 + d1 * d1)
      elif has_j_m1:
        d0 = float(val_j_m1) - mu_xy
        var_xy = d0 * d0
      elif has_i_m1:
        d1 = float(val_i_m1) - mu_xy
        var_xy = d1 * d1

    Dz = v_prev - mu_xy
    trust_z = (var_xy + eps) / (var_xy + (Dz * Dz) + eps)
    wz_f = float(wz_max) * trust_z
    pred_f = (wz_f * mu_z + float(wxy) * mu_xy) / (wz_f + float(wxy))
    return int(round(pred_f))

  if variant_id == 8:
    if k == 0:
      if c_intra == 0:
        return 0
      return _round_half_up_nonneg(s_intra, c_intra)

    v_prev = src[i, j, k - 1]
    if c_intra == 0:
      N_tau = eps_num
      D_tau = eps_num + eps_den * v_prev * v_prev
      if D_tau == 0:
        N_tau = 1
        D_tau = 1
      g = _gcd_int64(N_tau, D_tau)
      N_tau //= g
      D_tau //= g
      num = wz_max * N_tau * s_inter
      den = (wz_max * N_tau + wxy * D_tau) * c_inter
      return _round_half_up_nonneg(num, den)

    if var_win == 4:
      m_v = c_intra
      V_num = 0
      if has_j_m1:
        d0 = c_intra * val_j_m1 - s_intra
        V_num += d0 * d0
      if has_i_m1:
        d1 = c_intra * val_i_m1 - s_intra
        V_num += d1 * d1
      if has_diag_ul:
        d2 = c_intra * val_diag_ul - s_intra
        V_num += d2 * d2
      if has_diag_ur:
        d3 = c_intra * val_diag_ur - s_intra
        V_num += d3 * d3
    else:
      if has_j_m1 and has_i_m1:
        m_v = 2
        d0 = c_intra * val_j_m1 - s_intra
        d1 = c_intra * val_i_m1 - s_intra
        V_num = d0 * d0 + d1 * d1
      elif has_j_m1:
        m_v = 1
        d0 = c_intra * val_j_m1 - s_intra
        V_num = d0 * d0
      elif has_i_m1:
        m_v = 1
        d1 = c_intra * val_i_m1 - s_intra
        V_num = d1 * d1
      else:
        m_v = 1
        V_num = 0

    delta_z = c_intra * v_prev - s_intra
    c_sq = c_intra * c_intra

    N_tau = eps_den * V_num + eps_num * m_v * c_sq
    D_tau = N_tau + eps_den * m_v * (delta_z * delta_z)
    if D_tau == 0:
      N_tau = 1
      D_tau = 1
    else:
      g = _gcd_int64(N_tau, D_tau)
      if g > 1:
        N_tau //= g
        D_tau //= g

    max_s_term = max(s_inter * c_intra, max(s_intra * c_inter, c_inter * c_intra))
    safe_limit = INT64_MAX // (4 * max(wz_max + wxy, 1) * max(max_s_term, 1))
    if D_tau <= safe_limit:
      num = wz_max * N_tau * s_inter * c_intra + wxy * D_tau * s_intra * c_inter
      den = (wz_max * N_tau + wxy * D_tau) * c_inter * c_intra
      return _round_half_up_nonneg(num, den)
    else:
      return _awcp_v1_int_fallback_u128(
          N_tau, D_tau, wz_max, wxy, s_inter, c_inter, s_intra, c_intra
      )

  return 0


@njit(cache=True)
def _causal_scan_volume(
    arr_in: np.ndarray,
    is_recovery: bool,
    variant_id: int,
    wz_max: int,
    wxy: int,
    eps: float,
    eps_num: int,
    eps_den: int,
    var_win: int,
) -> np.ndarray:
  W, H, D = arr_in.shape
  out = np.zeros((W, H, D), dtype=np.int64)
  for k in range(D):
    for i in range(W):
      for j in range(H):
        src = out if is_recovery else arr_in
        pred_val = _predict_single_voxel(
            src, i, j, k, W, H, D, variant_id, wz_max, wxy, eps, eps_num, eps_den, var_win
        )
        if is_recovery:
          out[i, j, k] = pred_val + arr_in[i, j, k]
        else:
          out[i, j, k] = pred_val
  return out


def predict_volume(
    vp: np.ndarray,
    predictor_name: str,
    wz: int = 1,
    wxy: int = 2,
    eps: float = 1e-4,
    var_win: int = 2,
) -> np.ndarray:
  if predictor_name not in VARIANT_IDS:
    raise ValueError(f"Unknown predictor_name: {predictor_name}")
  variant_id = VARIANT_IDS[predictor_name]
  eff_wz = 0 if predictor_name == "WCP_INSLICE" else int(wz)
  eps_num, eps_den = eps_to_rational(eps)
  vp_i64 = np.ascontiguousarray(vp, dtype=np.int64)
  return _causal_scan_volume(
      vp_i64, False, variant_id, eff_wz, int(wxy), float(eps), eps_num, eps_den, int(var_win)
  )


def recover_volume_from_residuals(
    residual_vol: np.ndarray,
    variant_id: int,
    wz_max: int,
    wxy: int,
    eps: float,
    var_win: int,
) -> np.ndarray:
  eps_num, eps_den = eps_to_rational(eps)
  res_i64 = np.ascontiguousarray(residual_vol, dtype=np.int64)
  return _causal_scan_volume(
      res_i64, True, int(variant_id), int(wz_max), int(wxy), float(eps), eps_num, eps_den, int(var_win)
  )


def predict_causal13(vp: np.ndarray, **_) -> np.ndarray:
  return predict_volume(vp, "Causal13")


def predict_med(vp: np.ndarray, **_) -> np.ndarray:
  return predict_volume(vp, "MED")


def predict_wcp(vp: np.ndarray, wz: int = 1, wxy: int = 2, fair_k0: bool = False, **_) -> np.ndarray:
  name = "WCP_FAIR" if fair_k0 else "WCP_PAPER"
  return predict_volume(vp, name, wz=int(wz), wxy=int(wxy))


def predict_awcp(
    vp: np.ndarray, wz_max: int = 1, wxy: int = 2, eps: float = 1e-4, var_win: int = 2, **_
) -> np.ndarray:
  return predict_volume(vp, "AWCP_V1", wz=int(wz_max), wxy=int(wxy), eps=eps, var_win=var_win)


def predict_awcp_int(
    vp: np.ndarray, wz_max: int = 1, wxy: int = 2, eps: float = 1e-4, var_win: int = 2, **_
) -> np.ndarray:
  return predict_volume(vp, "AWCP_V1_INT", wz=int(wz_max), wxy=int(wxy), eps=eps, var_win=var_win)


PREDICTORS = {
    name: (
        lambda vp, _n=name, wz=1, wxy=2, eps=1e-4, var_win=2, **kw: (
            predict_volume(vp, _n, wz=wz, wxy=wxy, eps=eps, var_win=var_win)
        )
    )
    for name in VARIANT_IDS
}


@njit(cache=True)
def _compute_tau_stats_numba(
    vp: np.ndarray, wz_max: int, eps_num: int, eps_den: int, var_win: int
) -> Tuple[float, float, float, float, int]:
  W, H, D = vp.shape
  if D <= 1:
    return 0.0, 0.0, 0.0, 0.0, 0

  total_voxels = W * H * (D - 1)
  cnt_lt_001 = 0
  cnt_gt_099 = 0
  sum_tau = 0.0
  n_fallback = 0

  for k in range(1, D):
    for i in range(W):
      for j in range(H):
        s_intra = 0
        c_intra = 0
        val_j_m1 = 0
        val_i_m1 = 0
        val_diag_ul = 0
        val_diag_ur = 0
        has_j_m1 = False
        has_i_m1 = False
        has_diag_ul = False
        has_diag_ur = False

        if j - 1 >= 0:
          val_j_m1 = vp[i, j - 1, k]
          s_intra += val_j_m1
          c_intra += 1
          has_j_m1 = True
        if i - 1 >= 0:
          val_i_m1 = vp[i - 1, j, k]
          s_intra += val_i_m1
          c_intra += 1
          has_i_m1 = True
          if j - 1 >= 0:
            val_diag_ul = vp[i - 1, j - 1, k]
            s_intra += val_diag_ul
            c_intra += 1
            has_diag_ul = True
          if j + 1 < H:
            val_diag_ur = vp[i - 1, j + 1, k]
            s_intra += val_diag_ur
            c_intra += 1
            has_diag_ur = True

        v_prev = vp[i, j, k - 1]
        if c_intra == 0:
          N_tau = eps_num
          D_tau = eps_num + eps_den * v_prev * v_prev
        else:
          if var_win == 4:
            m_v = c_intra
            V_num = 0
            if has_j_m1:
              d0 = c_intra * val_j_m1 - s_intra
              V_num += d0 * d0
            if has_i_m1:
              d1 = c_intra * val_i_m1 - s_intra
              V_num += d1 * d1
            if has_diag_ul:
              d2 = c_intra * val_diag_ul - s_intra
              V_num += d2 * d2
            if has_diag_ur:
              d3 = c_intra * val_diag_ur - s_intra
              V_num += d3 * d3
          else:
            if has_j_m1 and has_i_m1:
              m_v = 2
              d0 = c_intra * val_j_m1 - s_intra
              d1 = c_intra * val_i_m1 - s_intra
              V_num = d0 * d0 + d1 * d1
            elif has_j_m1:
              m_v = 1
              d0 = c_intra * val_j_m1 - s_intra
              V_num = d0 * d0
            elif has_i_m1:
              m_v = 1
              d1 = c_intra * val_i_m1 - s_intra
              V_num = d1 * d1
            else:
              m_v = 1
              V_num = 0
          delta_z = c_intra * v_prev - s_intra
          c_sq = c_intra * c_intra
          N_tau = eps_den * V_num + eps_num * m_v * c_sq
          D_tau = N_tau + eps_den * m_v * (delta_z * delta_z)

        if D_tau == 0:
          N_tau = 1
          D_tau = 1
        else:
          g = _gcd_int64(N_tau, D_tau)
          if g > 1:
            N_tau //= g
            D_tau //= g

        if 100 * N_tau < D_tau:
          cnt_lt_001 += 1
        if 100 * N_tau > 99 * D_tau:
          cnt_gt_099 += 1

        tau_val = float(N_tau) / float(D_tau)
        sum_tau += tau_val

        s_inter = 0
        c_inter = 0
        for di in range(-1, 2):
          ii = i + di
          if 0 <= ii < W:
            for dj in range(-1, 2):
              jj = j + dj
              if 0 <= jj < H:
                s_inter += vp[ii, jj, k - 1]
                c_inter += 1
        max_s_term = max(
            s_inter * max(c_intra, 1),
            max(s_intra * c_inter, c_inter * max(c_intra, 1)),
        )
        safe_limit = INT64_MAX // (4 * max(wz_max + 2, 1) * max(max_s_term, 1))
        if D_tau > safe_limit:
          n_fallback += 1

  frac_lt_001 = float(cnt_lt_001) / float(total_voxels)
  frac_gt_099 = float(cnt_gt_099) / float(total_voxels)
  mean_tau = sum_tau / float(total_voxels)
  mean_wz = float(wz_max) * mean_tau
  return frac_lt_001, frac_gt_099, mean_tau, mean_wz, n_fallback


def warmup_jit() -> None:
  dummy = np.zeros((4, 4, 3), dtype=np.int64)
  for vid in (0, 1, 2, 3, 4, 5, 8):
    _ = _causal_scan_volume(dummy, False, vid, 1, 2, 1e-4, 1, 10000, 2)
    _ = _causal_scan_volume(dummy, True, vid, 1, 2, 1e-4, 1, 10000, 2)
  _ = compute_bit_length_int64(dummy)
  _ = _compute_tau_stats_numba(dummy, 1, 1, 10000, 2)


def label_and_capacity(residual: int, b: int = B_WIDTH) -> Tuple[int, int, int, int, int]:
  mag = abs(int(residual))
  ell = effective_bit_length(mag)
  z = b - ell if ell > 0 else b
  s = 1 if residual < 0 else 0
  label = 2 * z + s
  cap = min(z + 1, b)
  return ell, z, s, label, cap


def compute_residual_fields(residuals: np.ndarray, b: int = B_WIDTH):
  residuals_i64 = np.asarray(residuals, dtype=np.int64)
  mag = np.abs(residuals_i64)
  ell = compute_bit_length_int64(mag)
  z = np.where(ell > 0, b - ell, b)
  s = (residuals_i64 < 0).astype(np.int64)
  label = 2 * z + s
  cap = np.minimum(z + 1, b)
  return ell, z, s, label, cap


@dataclass
class CapacitySummary:
  N: int
  b: int
  be: int
  CBEC: int
  CRDC: int
  CTEC: int
  CAUX: int
  CPEC: int
  tree_bits_len: int = 0
  label_bits_len: int = 0

  @property
  def ABEC(self):
    return self.CBEC / self.N

  @property
  def ARDC(self):
    return self.CRDC / self.N

  @property
  def ATEC(self):
    return self.CTEC / self.N

  @property
  def AAUX(self):
    return self.CAUX / self.N

  @property
  def APEC(self):
    return self.CPEC / self.N


def capacity_summary(
    cap: np.ndarray,
    be: int,
    aux_bits: int,
    b: int = B_WIDTH,
    tree_bits_len: int = 0,
    label_bits_len: int = 0,
) -> CapacitySummary:
  N = int(cap.size)
  CBEC = N * (b - be)
  CTEC = int(cap.sum())
  CRDC = CTEC - CBEC
  CPEC = CTEC - aux_bits
  return CapacitySummary(
      N=N,
      b=b,
      be=be,
      CBEC=CBEC,
      CRDC=CRDC,
      CTEC=CTEC,
      CAUX=aux_bits,
      CPEC=CPEC,
      tree_bits_len=tree_bits_len,
      label_bits_len=label_bits_len,
  )


def residual_interval_occupancy(residuals: np.ndarray) -> Dict[str, float]:
  mag = np.abs(np.asarray(residuals, dtype=np.int64)).reshape(-1)
  N = mag.size
  bounds = [(0, 0), (1, 1), (2, 3), (4, 7), (8, 15), (16, 31), (32, 63)]
  names = ["P0", "P1", "P2-3", "P4-7", "P8-15", "P16-31", "P32-63"]
  out = {}
  for name, (lo, hi) in zip(names, bounds):
    out[name] = float(np.sum((mag >= lo) & (mag <= hi))) / N
  out["P64+"] = float(np.sum(mag >= 64)) / N
  return out


class _Node:
  __slots__ = ("sym", "left", "right")

  def __init__(self, sym=None, left=None, right=None):
    self.sym, self.left, self.right = sym, left, right

  def is_leaf(self):
    return self.left is None and self.right is None


def huffman_build_tree_from_counts(counts: np.ndarray) -> Tuple[_Node, int]:
  heap = []
  counter = 0
  for sym, c in enumerate(counts):
    w = int(c)
    if w > 0:
      heap.append((w, counter, _Node(sym=int(sym))))
      counter += 1
  n_leaves = len(heap)
  if n_leaves == 0:
    return _Node(sym=0), 0
  heapq.heapify(heap)
  if n_leaves == 1:
    return heap[0][2], 1
  while len(heap) > 1:
    w1, _, n1 = heapq.heappop(heap)
    w2, _, n2 = heapq.heappop(heap)
    counter += 1
    heapq.heappush(heap, (w1 + w2, counter, _Node(left=n1, right=n2)))
  return heap[0][2], n_leaves


def huffman_build_tree(labels_flat: np.ndarray) -> _Node:
  labels_flat = np.asarray(labels_flat, dtype=np.int64)
  if labels_flat.size == 0:
    return _Node(sym=0)
  counts = np.bincount(labels_flat, minlength=1)
  root, _ = huffman_build_tree_from_counts(counts)
  return root


def build_codes(root: _Node) -> Dict[int, str]:
  codes: Dict[int, str] = {}
  if root.is_leaf():
    codes[root.sym] = "0"
    return codes

  def rec(node, code):
    if node.is_leaf():
      codes[node.sym] = code
    else:
      rec(node.left, code + "0")
      rec(node.right, code + "1")

  rec(root, "")
  return codes


def build_code_lengths(root: _Node) -> Dict[int, int]:
  lengths: Dict[int, int] = {}
  if root.is_leaf():
    lengths[root.sym] = 1
    return lengths

  def rec(node, depth: int):
    if node.is_leaf():
      lengths[node.sym] = depth
    else:
      rec(node.left, depth + 1)
      rec(node.right, depth + 1)

  rec(root, 0)
  return lengths


def compute_exact_aux_lengths_from_labels(
    labels: np.ndarray,
    symbol_bits: int = SYMBOL_BITS,
    header_bits: int = HEADER_BITS,
) -> Tuple[int, int, int, Dict[int, int], np.ndarray]:
  flat = np.asarray(labels, dtype=np.int64).reshape(-1)
  counts = np.bincount(flat, minlength=2 * B_WIDTH + 2)
  root, n_leaves = huffman_build_tree_from_counts(counts)
  code_lengths = build_code_lengths(root)

  if n_leaves <= 1:
    tree_len = 1 + symbol_bits
  else:
    tree_len = (symbol_bits + 2) * n_leaves - 1

  label_len = int(sum(int(counts[sym]) * length for sym, length in code_lengths.items()))
  caux = int(header_bits + tree_len + label_len)
  return caux, int(tree_len), int(label_len), code_lengths, counts


def capacity_only(
    vol: np.ndarray,
    predictor_name: str = "WCP_PAPER",
    wz: int = 1,
    wxy: int = 2,
    eps: float = 1e-4,
    var_win: int = 2,
    b: int = B_WIDTH,
    compute_tau_Metrics: bool = True,
) -> dict:
  t0 = time.perf_counter()
  vp, delta, be = preprocess_volume(vol)
  W, H, D = vp.shape
  N = W * H * D
  if N < 512 or be >= b:
    raise ValueError(f"Component infeasible: N={N} (req >=512), be={be} (req <{b}).")

  if predictor_name not in VARIANT_IDS:
    raise ValueError(f"Unknown predictor_name: {predictor_name}")

  eff_wz = 0 if predictor_name == "WCP_INSLICE" else int(wz)
  eff_wxy = int(wxy)
  eps_num, eps_den = eps_to_rational(eps)

  tp0 = time.perf_counter()
  pred = predict_volume(vp, predictor_name, wz=eff_wz, wxy=eff_wxy, eps=eps, var_win=var_win)
  pred_s = time.perf_counter() - tp0

  residuals3d = vp - pred
  ell3d, z3d, s3d, label3d, cap3d = compute_residual_fields(residuals3d, b)

  caux, tree_len, label_len, code_lengths, counts = compute_exact_aux_lengths_from_labels(label3d)
  CBEC = N * (b - be)
  if caux > CBEC:
    raise ValueError(f"Component infeasible: CAUX={caux} > CBEC={CBEC}.")

  cap = capacity_summary(
      cap3d, be, aux_bits=caux, b=b, tree_bits_len=tree_len, label_bits_len=label_len
  )
  occ = residual_interval_occupancy(residuals3d)

  abs_res = np.abs(residuals3d)
  mae = float(np.mean(abs_res))
  rmse = float(np.sqrt(np.mean(residuals3d.astype(np.float64) ** 2)))

  if compute_tau_Metrics and D > 1:
    tau_lt_001, tau_gt_099, mean_tau, mean_wz_raw, n_u128_fallback = (
        _compute_tau_stats_numba(vp, eff_wz, eps_num, eps_den, int(var_win))
    )
    if predictor_name in ("AWCP_V1", "AWCP_V1_INT", "AWCP"):
      eff_mean_wz = mean_wz_raw
    elif predictor_name == "WCP_INSLICE":
      eff_mean_wz = 0.0
    else:
      eff_mean_wz = float(eff_wz)
  else:
    tau_lt_001, tau_gt_099, mean_tau, eff_mean_wz, n_u128_fallback = 0.0, 0.0, 0.0, 0.0, 0

  M = int(vp.max())
  overflow_info = check_awcp_int64_overflow_bound(
      be, M, max(eff_wz, 1), eff_wxy, eps_num, eps_den, var_win=var_win
  )
  total_s = time.perf_counter() - t0

  return dict(
      predictor_name=predictor_name,
      shape=(W, H, D),
      N=N,
      be=be,
      M=M,
      delta=delta,
      cap=cap,
      ABEC=cap.ABEC,
      ARDC=cap.ARDC,
      ATEC=cap.ATEC,
      AAUX=cap.AAUX,
      APEC=cap.APEC,
      CBEC=cap.CBEC,
      CRDC=cap.CRDC,
      CTEC=cap.CTEC,
      CAUX=cap.CAUX,
      CPEC=cap.CPEC,
      tree_bits_len=tree_len,
      label_bits_len=label_len,
      n_unique_labels=len(code_lengths),
      MAE=mae,
      RMSE=rmse,
      occupancy=occ,
      P0=occ["P0"],
      P1=occ["P1"],
      P2_3=occ["P2-3"],
      P4_7=occ["P4-7"],
      P8_15=occ["P8-15"],
      P16_31=occ["P16-31"],
      P32_63=occ["P32-63"],
      P64_plus=occ["P64+"],
      tau_lt_001_frac=tau_lt_001,
      tau_gt_099_frac=tau_gt_099,
      tau_mean=mean_tau,
      wz_mean=eff_mean_wz,
      n_u128_fallback=n_u128_fallback,
      overflow_info=overflow_info,
      pred_s=pred_s,
      total_s=total_s,
  )


def serialize_tree(root: _Node, symbol_bits: int = SYMBOL_BITS) -> str:
  bits: List[str] = []

  def rec(node):
    if node.is_leaf():
      bits.append("1")
      bits.append(format(node.sym, f"0{symbol_bits}b"))
    else:
      bits.append("0")
      rec(node.left)
      rec(node.right)

  rec(root)
  return "".join(bits)


def deserialize_tree(bits: str, pos: int, symbol_bits: int = SYMBOL_BITS) -> Tuple[_Node, int]:
  bit = bits[pos]
  pos += 1
  if bit == "1":
    sym = int(bits[pos : pos + symbol_bits], 2)
    pos += symbol_bits
    return _Node(sym=sym), pos
  left, pos = deserialize_tree(bits, pos, symbol_bits)
  right, pos = deserialize_tree(bits, pos, symbol_bits)
  return _Node(left=left, right=right), pos


def huffman_encode(labels_flat: np.ndarray, codes: Dict[int, str]) -> str:
  max_sym = int(labels_flat.max()) if labels_flat.size else 0
  lut = [codes.get(i, "") for i in range(max_sym + 1)]
  return "".join(lut[s] for s in labels_flat)


def huffman_decode(bits: str, root: _Node, count: int) -> List[int]:
  out = []
  if root.is_leaf():
    return [root.sym] * count
  pos = 0
  node = root
  while len(out) < count:
    node = node.left if bits[pos] == "0" else node.right
    pos += 1
    if node.is_leaf():
      out.append(node.sym)
      node = root
  return out


def rdc_bit_positions(ell: int, be: int) -> List[int]:
  if ell == 0:
    return list(range(be - 1, -1, -1))
  return list(range(be - 1, ell - 2, -1))


def _embed_stream_into_planes(
    marked: np.ndarray,
    bits_arr: np.ndarray,
    planes: List[int],
    start_plane_idx: int = 0,
    start_voxel_idx: int = 0,
) -> Tuple[int, int]:
  N = marked.shape[0]
  ptr = 0
  plane_idx, voxel_idx = start_plane_idx, start_voxel_idx
  bits_arr = bits_arr.astype(np.uint64)
  while ptr < bits_arr.size and plane_idx < len(planes):
    plane = planes[plane_idx]
    take = min(N - voxel_idx, bits_arr.size - ptr)
    sl = slice(voxel_idx, voxel_idx + take)
    chunk = bits_arr[ptr : ptr + take]
    marked[sl] = (marked[sl] & ~np.uint64(1 << plane)) | (chunk << np.uint64(plane))
    ptr += take
    voxel_idx += take
    if voxel_idx >= N:
      plane_idx += 1
      voxel_idx = 0
  return plane_idx, voxel_idx


def _read_stream_from_planes(
    marked: np.ndarray,
    nbits: int,
    planes: List[int],
    start_plane_idx: int = 0,
    start_voxel_idx: int = 0,
) -> Tuple[np.ndarray, int, int]:
  N = marked.shape[0]
  out = np.zeros(nbits, dtype=np.uint8)
  got = 0
  plane_idx, voxel_idx = start_plane_idx, start_voxel_idx
  while got < nbits and plane_idx < len(planes):
    plane = planes[plane_idx]
    plane_bits = ((marked >> np.uint64(plane)) & np.uint64(1)).astype(np.uint8)
    take = min(N - voxel_idx, nbits - got)
    out[got : got + take] = plane_bits[voxel_idx : voxel_idx + take]
    got += take
    voxel_idx += take
    if voxel_idx >= N:
      plane_idx += 1
      voxel_idx = 0
  return out, plane_idx, voxel_idx


def _embed_stream_into_rdc(
    marked: np.ndarray, bits_arr: np.ndarray, ell_flat: np.ndarray, be: int
) -> int:
  ptr = 0
  bits_arr = bits_arr.astype(np.uint64)
  for plane in range(be - 1, -1, -1):
    if ptr >= bits_arr.size:
      break
    eligible = (ell_flat == 0) | (plane >= (ell_flat - 1))
    idxs = np.nonzero(eligible)[0]
    take = min(idxs.size, bits_arr.size - ptr)
    if take <= 0:
      continue
    chosen = idxs[:take]
    chunk = bits_arr[ptr : ptr + take]
    ptr += take
    marked[chosen] = (marked[chosen] & ~np.uint64(1 << plane)) | (chunk << np.uint64(plane))
  return ptr


def _read_stream_from_rdc(
    marked: np.ndarray, nbits: int, ell_flat: np.ndarray, be: int
) -> Tuple[np.ndarray, int]:
  out = np.zeros(nbits, dtype=np.uint8)
  got = 0
  for plane in range(be - 1, -1, -1):
    if got >= nbits:
      break
    eligible = (ell_flat == 0) | (plane >= (ell_flat - 1))
    idxs = np.nonzero(eligible)[0]
    take = min(idxs.size, nbits - got)
    if take <= 0:
      continue
    chosen = idxs[:take]
    bits = ((marked[chosen] >> np.uint64(plane)) & np.uint64(1)).astype(np.uint8)
    out[got : got + take] = bits
    got += take
  return out, got


def _bitstring_to_arr(s: str) -> np.ndarray:
  if not s:
    return np.zeros(0, dtype=np.uint8)
  return np.frombuffer(s.encode("ascii"), dtype=np.uint8) - ord("0")


def _arr_to_bitstring(a: np.ndarray) -> str:
  if a.size == 0:
    return ""
  return (a.astype(np.uint8) + ord("0")).tobytes().decode("ascii")


def generate_keystream(key: bytes, iv16: bytes, nbytes: int) -> bytes:
  assert len(key) == 32
  assert len(iv16) == 16
  cipher = Cipher(algorithms.AES(key), modes.CTR(iv16))
  encryptor = cipher.encryptor()
  return encryptor.update(b"\x00" * nbytes) + encryptor.finalize()


def derive_data_hiding_iv(iv_e: bytes) -> bytes:
  return bytes([iv_e[0] ^ 0xA5]) + iv_e[1:]


def _to_bits(value: int, width: int) -> str:
  assert 0 <= value < (1 << width), f"value={value} out of range for {width} bits"
  return format(value, f"0{width}b")


def pack_reserved_54bits(
    variant_id: int = 0,
    wz_max: int = 1,
    wxy: int = 2,
    eps_code: int = 4,
    var_win_code: int = 0,
    label_ctx_flag: int = 0,
    ablation_flags: int = 0,
) -> str:
  parts = [
      _to_bits(variant_id, 6),
      _to_bits(wz_max, 8),
      _to_bits(wxy, 8),
      _to_bits(eps_code, 4),
      _to_bits(var_win_code, 1),
      _to_bits(label_ctx_flag, 2),
      _to_bits(ablation_flags, 11),
      "0" * 14,
  ]
  res = "".join(parts)
  assert len(res) == 54
  return res


def unpack_reserved_54bits(bits54: str) -> dict:
  assert len(bits54) == 54
  pos = 0

  def take(n):
    nonlocal pos
    c = bits54[pos : pos + n]
    pos += n
    return int(c, 2)

  variant_id = take(6)
  wz_max = take(8)
  wxy = take(8)
  eps_code = take(4)
  var_win_code = take(1)
  label_ctx_flag = take(2)
  ablation_flags = take(11)
  _pad = take(14)
  return dict(
      variant_id=variant_id,
      wz_max=wz_max,
      wxy=wxy,
      eps_code=eps_code,
      eps=code_to_eps(eps_code),
      var_win=4 if var_win_code == 1 else 2,
      label_ctx_flag=label_ctx_flag,
      ablation_flags=ablation_flags,
  )


def build_bootstrap_header(
    b: int,
    be: int,
    predictor_id: int,
    symbol_size: int,
    offset: float,
    N: int,
    tree_len: int,
    label_len: int,
    secret_len: int,
    aux_len: int,
    iv_e: bytes,
    variant_id: int = 0,
    wz_max: int = 1,
    wxy: int = 2,
    eps_code: int = 4,
    var_win_code: int = 0,
    label_ctx_flag: int = 0,
    ablation_flags: int = 0,
) -> str:
  offset_bits = _to_bits(struct.unpack(">I", struct.pack(">f", offset))[0], 32)
  iv_bits = "".join(_to_bits(byte, 8) for byte in iv_e)
  reserved_54 = pack_reserved_54bits(
      variant_id=variant_id,
      wz_max=wz_max,
      wxy=wxy,
      eps_code=eps_code,
      var_win_code=var_win_code,
      label_ctx_flag=label_ctx_flag,
      ablation_flags=ablation_flags,
  )
  fields = [
      _to_bits(0xA5C3, 16),
      _to_bits(1, 8),
      _to_bits(b, 6),
      _to_bits(be, 6),
      _to_bits(predictor_id, 2),
      _to_bits(symbol_size, 4),
      offset_bits,
      _to_bits(N, 64),
      _to_bits(tree_len, 48),
      _to_bits(label_len, 48),
      _to_bits(secret_len, 48),
      _to_bits(aux_len, 48),
      iv_bits,
      reserved_54,
  ]
  header = "".join(fields)
  assert len(header) == HEADER_BITS
  return header


def parse_bootstrap_header(bits: str) -> dict:
  pos = 0

  def take(n):
    nonlocal pos
    chunk = bits[pos : pos + n]
    pos += n
    return chunk

  fmt_id = int(take(16), 2)
  version = int(take(8), 2)
  b = int(take(6), 2)
  be = int(take(6), 2)
  predictor_id = int(take(2), 2)
  symbol_size = int(take(4), 2)
  offset = struct.unpack(">f", struct.pack(">I", int(take(32), 2)))[0]
  N = int(take(64), 2)
  tree_len = int(take(48), 2)
  label_len = int(take(48), 2)
  secret_len = int(take(48), 2)
  aux_len = int(take(48), 2)
  iv_bits = take(128)
  iv_e = bytes(int(iv_bits[i : i + 8], 2) for i in range(0, 128, 8))
  reserved_54 = take(54)
  assert fmt_id == 0xA5C3, f"bad bootstrap header format ID: {hex(fmt_id)}"

  ext = unpack_reserved_54bits(reserved_54)
  out = dict(
      version=version,
      b=b,
      be=be,
      predictor_id=predictor_id,
      symbol_size=symbol_size,
      offset=offset,
      N=N,
      tree_len=tree_len,
      label_len=label_len,
      secret_len=secret_len,
      aux_len=aux_len,
      iv_e=iv_e,
  )
  out.update(ext)
  return out


@dataclass
class RDHBundle:
  marked_words: np.ndarray
  shape: Tuple[int, int, int]
  _key_e: Optional[bytes] = field(default=None, repr=False)
  _key_d: Optional[bytes] = field(default=None, repr=False)


def owner_embed(
    vol: np.ndarray,
    predictor_name: str,
    key_e: bytes,
    key_d: bytes,
    iv_e: bytes,
    secret_bits: str,
    wz: int = 1,
    wxy: int = 2,
    eps: float = 1e-4,
    var_win: int = 2,
    label_ctx_flag: int = 0,
    ablation_flags: int = 0,
    b: int = B_WIDTH,
) -> Tuple[RDHBundle, CapacitySummary, dict]:
  vp, delta, be = preprocess_volume(vol)
  W, H, D = vp.shape
  N = W * H * D
  if N < 512 or be >= b:
    raise ValueError(f"Component infeasible before embedding: N={N} (req >=512), be={be} (req <{b}).")

  if predictor_name not in VARIANT_IDS:
    raise ValueError(f"Unknown predictor_name: {predictor_name}")
  variant_id = VARIANT_IDS[predictor_name]
  predictor_id = PREDICTOR_FAMILY_IDS[predictor_name]
  eff_wz = 0 if predictor_name == "WCP_INSLICE" else int(wz)
  eff_wxy = int(wxy)
  eps_code = eps_to_code(eps)
  var_win_code = 1 if int(var_win) == 4 else 0

  pred = predict_volume(vp, predictor_name, wz=eff_wz, wxy=eff_wxy, eps=eps, var_win=var_win)
  residuals3d = vp - pred

  ell3d, z3d, s3d, label3d, cap3d = compute_residual_fields(residuals3d, b)

  labels_flat = _flatten_kij(label3d).astype(np.int64)
  ell_flat = _flatten_kij(ell3d).astype(np.int64)
  mag_flat = np.abs(_flatten_kij(residuals3d)).astype(np.int64)

  tree = huffman_build_tree(labels_flat)
  codes = build_codes(tree)
  tree_bits = serialize_tree(tree)
  label_bits = huffman_encode(labels_flat, codes)

  aux_len_final = HEADER_BITS + len(tree_bits) + len(label_bits)
  header_bits = build_bootstrap_header(
      b=b,
      be=be,
      predictor_id=predictor_id,
      symbol_size=SYMBOL_BITS,
      offset=delta,
      N=N,
      tree_len=len(tree_bits),
      label_len=len(label_bits),
      secret_len=len(secret_bits),
      aux_len=aux_len_final,
      iv_e=iv_e,
      variant_id=variant_id,
      wz_max=eff_wz,
      wxy=eff_wxy,
      eps_code=eps_code,
      var_win_code=var_win_code,
      label_ctx_flag=label_ctx_flag,
      ablation_flags=ablation_flags,
  )
  aux_bits = header_bits + tree_bits + label_bits

  CBEC = N * (b - be)
  if len(aux_bits) > CBEC:
    raise ValueError(f"Component infeasible: |aux|={len(aux_bits)} > CBEC={CBEC}.")

  keystream = generate_keystream(key_e, iv_e, N * 4)
  kwords = np.frombuffer(keystream, dtype=">u4").astype(np.uint64)
  marked = mag_flat.astype(np.uint64) ^ kwords

  bec_planes = list(range(b - 1, be - 1, -1))
  aux_arr = _bitstring_to_arr(aux_bits)
  plane_idx, voxel_idx = _embed_stream_into_planes(marked, aux_arr, bec_planes)

  secret_raw_arr = _bitstring_to_arr(secret_bits)
  if secret_raw_arr.size > 0:
    key_d_iv = derive_data_hiding_iv(iv_e)
    secret_ks = generate_keystream(key_d, key_d_iv, (secret_raw_arr.size + 7) // 8 + 4)
    ks_bits_arr = np.unpackbits(np.frombuffer(secret_ks, dtype=np.uint8))[: secret_raw_arr.size]
    secret_arr = (secret_raw_arr ^ ks_bits_arr).astype(np.uint8)
  else:
    secret_arr = secret_raw_arr

  total_bec = N * (b - be)
  rem_bec = total_bec - len(aux_bits)
  n_bec_secret = min(secret_arr.size, rem_bec)
  if n_bec_secret > 0:
    plane_idx, voxel_idx = _embed_stream_into_planes(
        marked, secret_arr[:n_bec_secret], bec_planes, plane_idx, voxel_idx
    )

  remaining_secret = secret_arr[n_bec_secret:]
  if remaining_secret.size > 0:
    written = _embed_stream_into_rdc(marked, remaining_secret, ell_flat, be)
    if written < remaining_secret.size:
      raise ValueError("Secret payload exceeds pure embedding capacity (CPEC).")

  cap_summary = capacity_summary(
      cap3d,
      be,
      aux_bits=len(aux_bits),
      b=b,
      tree_bits_len=len(tree_bits),
      label_bits_len=len(label_bits),
  )
  bundle = RDHBundle(
      marked_words=marked.astype(np.uint32),
      shape=(W, H, D),
      _key_e=key_e,
      _key_d=key_d,
  )
  debug = dict(
      header_bits=header_bits,
      tree_bits=tree_bits,
      label_bits=label_bits,
      ell_flat=ell_flat,
      occupancy=residual_interval_occupancy(residuals3d),
      header_meta=parse_bootstrap_header(header_bits),
  )
  return bundle, cap_summary, debug


def _read_bootstrap_header_from_marked(marked: np.ndarray) -> dict:
  if marked.shape[0] < HEADER_BITS:
    raise ValueError(f"Volume has N={marked.shape[0]} < {HEADER_BITS}; cannot read bootstrap header.")
  msb_plane = np.uint64(B_WIDTH - 1)
  header_arr = ((marked[:HEADER_BITS] >> msb_plane) & np.uint64(1)).astype(np.uint8)
  return parse_bootstrap_header(_arr_to_bitstring(header_arr))


def receiver_extract_secret(bundle: RDHBundle, key_d: Optional[bytes] = None) -> str:
  use_key_d = key_d if key_d is not None else bundle._key_d
  if use_key_d is None:
    raise ValueError("key_d must be provided to receiver_extract_secret.")

  marked = bundle.marked_words.astype(np.uint64)
  meta = _read_bootstrap_header_from_marked(marked)
  N, b, be = meta["N"], meta["b"], meta["be"]
  tree_len, label_len, secret_len = meta["tree_len"], meta["label_len"], meta["secret_len"]
  if secret_len == 0:
    return ""

  bec_planes = list(range(b - 1, be - 1, -1))
  plane_idx = HEADER_BITS // N
  voxel_idx = HEADER_BITS % N

  rest_arr, plane_idx, voxel_idx = _read_stream_from_planes(
      marked, tree_len + label_len, bec_planes, plane_idx, voxel_idx
  )
  rest_bits = _arr_to_bitstring(rest_arr)
  tree_bits, label_bits = rest_bits[:tree_len], rest_bits[tree_len : tree_len + label_len]
  tree_root, _ = deserialize_tree(tree_bits, 0, meta["symbol_size"])
  labels = huffman_decode(label_bits, tree_root, N)
  labels_arr = np.array(labels, dtype=np.int64)
  z = labels_arr // 2
  ell_flat = np.where(z == b, 0, b - z)

  aux_total_len = HEADER_BITS + tree_len + label_len
  total_bec = N * (b - be)
  rem_bec = total_bec - aux_total_len
  n_bec_secret = min(secret_len, rem_bec)
  sec_bec_arr, _, _ = _read_stream_from_planes(
      marked, n_bec_secret, bec_planes, plane_idx, voxel_idx
  )

  if secret_len > n_bec_secret:
    need = secret_len - n_bec_secret
    rdc_arr, got = _read_stream_from_rdc(marked, need, ell_flat, be)
    whitened_arr = np.concatenate([sec_bec_arr, rdc_arr[:got]])
  else:
    whitened_arr = sec_bec_arr

  key_d_iv = derive_data_hiding_iv(meta["iv_e"])
  ks = generate_keystream(use_key_d, key_d_iv, (secret_len + 7) // 8 + 4)
  ks_bits_arr = np.unpackbits(np.frombuffer(ks, dtype=np.uint8))[:secret_len]
  secret_arr = (whitened_arr ^ ks_bits_arr).astype(np.uint8)
  return _arr_to_bitstring(secret_arr)


def receiver_recover_volume(bundle: RDHBundle, key_e: Optional[bytes] = None) -> np.ndarray:
  use_key_e = key_e if key_e is not None else bundle._key_e
  if use_key_e is None:
    raise ValueError("key_e must be provided to receiver_recover_volume.")

  marked = bundle.marked_words.astype(np.uint64)
  W, H, D = bundle.shape
  meta = _read_bootstrap_header_from_marked(marked)
  N, b, be = meta["N"], meta["b"], meta["be"]
  assert N == W * H * D, f"Header N={N} does not match container shape {bundle.shape}"
  tree_len, label_len = meta["tree_len"], meta["label_len"]

  bec_planes = list(range(b - 1, be - 1, -1))
  plane_idx = HEADER_BITS // N
  voxel_idx = HEADER_BITS % N

  rest_arr, _, _ = _read_stream_from_planes(
      marked, tree_len + label_len, bec_planes, plane_idx, voxel_idx
  )
  rest_bits = _arr_to_bitstring(rest_arr)
  tree_bits, label_bits = rest_bits[:tree_len], rest_bits[tree_len : tree_len + label_len]
  tree_root, _ = deserialize_tree(tree_bits, 0, meta["symbol_size"])
  labels = huffman_decode(label_bits, tree_root, N)
  labels_arr = np.array(labels, dtype=np.int64)

  keystream = generate_keystream(use_key_e, meta["iv_e"], N * 4)
  kwords = np.frombuffer(keystream, dtype=">u4").astype(np.uint64)
  decrypted = marked ^ kwords

  z = labels_arr // 2
  s = labels_arr % 2
  ell = np.where(z == b, 0, b - z)
  ell_safe = np.where(ell > 0, ell, 1).astype(np.uint64)
  mask = np.left_shift(np.uint64(1), ell_safe - np.uint64(1)) - np.uint64(1)
  low_bits = decrypted & mask
  mag_u64 = np.where(
      ell > 0,
      np.left_shift(np.uint64(1), ell_safe - np.uint64(1)) + low_bits,
      np.uint64(0),
  )
  mag = mag_u64.astype(np.int64)
  signed_residual_flat = np.where(s == 1, -mag, mag)

  residual_vol = _unflatten_kij(signed_residual_flat, (W, H, D))
  vprime_rec = recover_volume_from_residuals(
      residual_vol=residual_vol,
      variant_id=meta["variant_id"],
      wz_max=meta["wz_max"],
      wxy=meta["wxy"],
      eps=meta["eps"],
      var_win=meta["var_win"],
  )

  v_rec = (vprime_rec.astype(np.float64) + meta["offset"]).astype(np.float32)
  return v_rec


def evaluate_volume_pipeline(
    vol: np.ndarray,
    predictor_name: str = "WCP_PAPER",
    payload_fraction: float = 0.90,
    wz: int = 1,
    wxy: int = 2,
    eps: float = 1e-4,
    var_win: int = 2,
    seed: int = 0,
    n_pred_repeats: int = 5,
) -> dict:
  rng = np.random.default_rng(seed)
  key_e = bytes(rng.integers(0, 256, size=32, dtype=np.uint8).tolist())
  key_d = bytes(rng.integers(0, 256, size=32, dtype=np.uint8).tolist())
  iv_e = bytes(rng.integers(0, 256, size=16, dtype=np.uint8).tolist())

  vp, _, _ = preprocess_volume(vol)
  pred_times = []
  for _ in range(max(5, int(n_pred_repeats))):
    tp0 = time.perf_counter()
    _ = predict_volume(vp, predictor_name, wz=wz, wxy=wxy, eps=eps, var_win=var_win)
    pred_times.append(time.perf_counter() - tp0)
  time_pred_median_s = float(np.median(pred_times))

  probe = capacity_only(
      vol, predictor_name=predictor_name, wz=wz, wxy=wxy, eps=eps, var_win=var_win
  )
  n_secret_bits = max(0, min(int(probe["CPEC"] * payload_fraction), probe["CPEC"]))
  secret_arr = rng.integers(0, 2, size=n_secret_bits, dtype=np.uint8)
  secret_bits = _arr_to_bitstring(secret_arr)

  t0 = time.perf_counter()
  bundle, cap, dbg = owner_embed(
      vol,
      predictor_name,
      key_e,
      key_d,
      iv_e,
      secret_bits,
      wz=wz,
      wxy=wxy,
      eps=eps,
      var_win=var_win,
  )
  t1 = time.perf_counter()
  extracted = receiver_extract_secret(bundle, key_d=key_d)
  t2 = time.perf_counter()
  recovered = receiver_recover_volume(bundle, key_e=key_e)
  t3 = time.perf_counter()

  assert cap.CAUX == probe["CAUX"] and cap.CPEC == probe["CPEC"], (
      f"Mismatch between capacity_only ({probe['CAUX']}, {probe['CPEC']}) "
      f"and owner_embed ({cap.CAUX}, {cap.CPEC})!"
  )

  if n_secret_bits > 0:
    ext_arr = _bitstring_to_arr(extracted)
    ber = float(np.mean(secret_arr != ext_arr)) if ext_arr.size == secret_arr.size else 1.0
  else:
    ber = 0.0

  vol32 = vol.astype(np.float32)
  exact_numeric = bool(np.array_equal(recovered, vol32))
  exact_bits = bool(np.array_equal(recovered.view(np.uint32), vol32.view(np.uint32)))

  assert ber == 0.0, f"BER={ber} != 0 on {predictor_name}"
  assert exact_numeric and exact_bits, f"Bit-exact recovery failed on {predictor_name}"

  return dict(
      shape=vol.shape,
      cap=cap,
      ber=ber,
      exact_numeric=exact_numeric,
      exact_bits=exact_bits,
      occupancy=dbg["occupancy"],
      probe=probe,
      timings=dict(
          pred_kernel_median_s=time_pred_median_s,
          embed_s=t1 - t0,
          extract_s=t2 - t1,
          recover_s=t3 - t2,
          pipeline_total_s=(t3 - t0),
      ),
  )

Overwriting rdh_encrypted_3d_medical_volumes.py


In [ ]:
import os

print(os.path.exists("rdh_encrypted_3d_medical_volumes.py"))
# Output harus True

True


Sesi 2

In [ ]:
%%writefile run_stage1.py
"""
Stage 0 + Stage 1 Complete Verification & Prostate Evaluation Suite.
Fixes the dictionary key typo ('P16_31' and 'P32_63') and prints any
exception explicitly so no failure is ever hidden.
"""

import importlib
import os
import numpy as np
import pandas as pd
import rdh_encrypted_3d_medical_volumes as rdh

rdh = importlib.reload(rdh)

STAGE1_PREDICTORS = [
    "Causal13_FLOAT",
    "Causal13",
    "MED",
    "WCP_PAPER_FLOAT",
    "WCP_PAPER",
    "WCP_FAIR",
    "WCP_INSLICE",
    "AWCP_V1",
    "AWCP_V1_INT",
]

METRIC_COLUMNS = [
    "Filename",
    "Component",
    "Predictor",
    "status",
    "error_message",
    "Shape",
    "N",
    "be",
    "ABEC",
    "ARDC",
    "ATEC",
    "AAUX",
    "Tree_Bits",
    "Label_Bits",
    "APEC",
    "CPEC",
    "MAE",
    "RMSE",
    "P0",
    "P1",
    "P2_3",
    "P4_7",
    "P8_15",
    "P16_31",
    "P32_63",
    "P64+",
    "tau_lt_001",
    "tau_gt_099",
    "tau_mean",
    "wz_mean",
    "u128_fallbacks",
    "Pred_s",
]


def run_stage1d_edge_and_security_tests():
  print("=" * 90)
  print("TAHAP 1.d: UJI KASUS TEPI (EDGE CASES), OVERFLOW, DAN KEAMANAN KUNCI")
  print("=" * 90)
  rdh.warmup_jit()
  rng = np.random.default_rng(123)
  key_e = bytes(rng.integers(0, 256, size=32, dtype=np.uint8).tolist())
  key_d = bytes(rng.integers(0, 256, size=32, dtype=np.uint8).tolist())
  iv_e = bytes(rng.integers(0, 256, size=16, dtype=np.uint8).tolist())
  payload = "1100101011110000" * 32  # 512 bits

  # 1. Negative HU (offset delta < 0)
  vol_neg = rng.integers(-1024, 1500, size=(16, 16, 4)).astype(np.float32)
  vp_neg, delta_neg, be_neg = rdh.preprocess_volume(vol_neg)
  assert delta_neg < 0, f"Expected negative delta, got {delta_neg}"
  for pname in ("WCP_FAIR", "AWCP_V1", "AWCP_V1_INT"):
    c_fast = rdh.capacity_only(vol_neg, pname, wz=1, wxy=2)
    bundle, cap, _ = rdh.owner_embed(
        vol_neg, pname, key_e, key_d, iv_e, payload, wz=1, wxy=2
    )
    assert c_fast["CAUX"] == cap.CAUX and c_fast["CPEC"] == cap.CPEC
    rec = rdh.receiver_recover_volume(bundle, key_e=key_e)
    assert np.array_equal(rec.view(np.uint32), vol_neg.view(np.uint32))
  print(
      f"[LULUS 1/7] HU Negatif (delta={delta_neg:.1f}, be={be_neg}):"
      " capacity_only == owner_embed & BER=0."
  )

  # 2. Single slice D = 1 (N = 32*32*1 = 1024 >= 512)
  vol_d1 = rng.integers(0, 500, size=(32, 32, 1)).astype(np.float32)
  for pname in ("Causal13", "MED", "WCP_PAPER", "WCP_FAIR", "WCP_INSLICE", "AWCP_V1", "AWCP_V1_INT"):
    c_fast = rdh.capacity_only(vol_d1, pname, wz=1, wxy=2)
    bundle, cap, _ = rdh.owner_embed(
        vol_d1, pname, key_e, key_d, iv_e, payload, wz=1, wxy=2
    )
    assert c_fast["CAUX"] == cap.CAUX and c_fast["CPEC"] == cap.CPEC
    rec = rdh.receiver_recover_volume(bundle, key_e=key_e)
    assert np.array_equal(rec.view(np.uint32), vol_d1.view(np.uint32))
  print(
      "[LULUS 2/7] Volume D=1 (32x32x1): semua 7 prediktor identik antara"
      " capacity_only & pipeline penuh."
  )

  # 3. Constant volume (Single Huffman symbol n = 1 -> tree_len = 8, label_len = N)
  vol_zero = np.zeros((16, 16, 4), dtype=np.float32)
  c_zero = rdh.capacity_only(vol_zero, "AWCP_V1_INT", wz=1, wxy=2)
  b_zero, cap_zero, _ = rdh.owner_embed(
      vol_zero, "AWCP_V1_INT", key_e, key_d, iv_e, payload, wz=1, wxy=2
  )
  assert c_zero["n_unique_labels"] == 1
  assert c_zero["tree_bits_len"] == 8
  assert c_zero["label_bits_len"] == vol_zero.size
  assert c_zero["CAUX"] == cap_zero.CAUX and c_zero["CPEC"] == cap_zero.CPEC
  rec_zero = rdh.receiver_recover_volume(b_zero, key_e=key_e)
  assert np.array_equal(rec_zero.view(np.uint32), vol_zero.view(np.uint32))
  print(
      "[LULUS 3/7] Volume Konstan 1-Simbol (n=1): tree_len=8 bit,"
      f" label_len={vol_zero.size} bit, BER=0."
  )

  # 4. Effective bit-depth be = 1 (values in {0, 1})
  vol_be1 = rng.integers(0, 2, size=(16, 16, 4)).astype(np.float32)
  c_be1 = rdh.capacity_only(vol_be1, "AWCP_V1_INT", wz=1, wxy=2)
  b_be1, cap_be1, _ = rdh.owner_embed(
      vol_be1, "AWCP_V1_INT", key_e, key_d, iv_e, payload, wz=1, wxy=2
  )
  assert c_be1["be"] == 1 and cap_be1.ABEC == 31.0
  assert c_be1["CAUX"] == cap_be1.CAUX and c_be1["CPEC"] == cap_be1.CPEC
  rec_be1 = rdh.receiver_recover_volume(b_be1, key_e=key_e)
  assert np.array_equal(rec_be1.view(np.uint32), vol_be1.view(np.uint32))
  print(
      f"[LULUS 4/7] Volume be=1 (ABEC={cap_be1.ABEC:.1f}): capacity_only =="
      " owner_embed & pemulihan eksak."
  )

  # 5. Infeasible volume N < 512 (must raise ValueError)
  vol_small = rng.integers(0, 100, size=(8, 8, 4)).astype(np.float32)
  raised_fast = False
  raised_embed = False
  try:
    rdh.capacity_only(vol_small, "WCP_FAIR")
  except ValueError:
    raised_fast = True
  try:
    rdh.owner_embed(vol_small, "WCP_FAIR", key_e, key_d, iv_e, "")
  except ValueError:
    raised_embed = True
  assert raised_fast and raised_embed
  print(
      "[LULUS 5/7] Volume N<512 (N=256): ditolak dengan ValueError sesuai"
      " syarat kelayakan."
  )

  # 6. Wrong K_E
  vol_sec = rng.integers(0, 1000, size=(16, 16, 4)).astype(np.float32)
  sec_msg = "".join(str(x) for x in rng.integers(0, 2, size=4000))
  bundle_sec, _, _ = rdh.owner_embed(
      vol_sec, "AWCP_V1_INT", key_e, key_d, iv_e, sec_msg, wz=1, wxy=2
  )
  wrong_key_e = bytes([b ^ 0xFF for b in key_e])
  rec_wrong_ke = rdh.receiver_recover_volume(bundle_sec, key_e=wrong_key_e)
  ext_right_kd = rdh.receiver_extract_secret(bundle_sec, key_d=key_d)
  assert not np.array_equal(rec_wrong_ke, vol_sec)
  assert ext_right_kd == sec_msg
  print(
      "[LULUS 6/7] Kunci K_E Salah: pemulihan volume gagal total, ekstraksi"
      " pesan tetap BER=0."
  )

  # 7. Wrong K_D
  wrong_key_d = bytes([b ^ 0xAA for b in key_d])
  ext_wrong_kd = rdh.receiver_extract_secret(bundle_sec, key_d=wrong_key_d)
  ber_wrong_kd = float(np.mean([a != b for a, b in zip(sec_msg, ext_wrong_kd)]))
  rec_right_ke = rdh.receiver_recover_volume(bundle_sec, key_e=key_e)
  assert 0.42 <= ber_wrong_kd <= 0.58
  assert np.array_equal(rec_right_ke.view(np.uint32), vol_sec.view(np.uint32))
  print(
      "[LULUS 7/7] Kunci K_D Salah: BER pesan ="
      f" {ber_wrong_kd:.4f} (~0.5), pemulihan volume tetap BitExact=True."
  )

  # Overflow 128-bit fallback test
  vol_hi = rng.integers(0, 1 << 19, size=(12, 12, 4)).astype(np.float32)
  c_hi = rdh.capacity_only(vol_hi, "AWCP_V1_INT", wz=4, wxy=2, eps=1e-8)
  b_hi, _, _ = rdh.owner_embed(
      vol_hi, "AWCP_V1_INT", key_e, key_d, iv_e, payload, wz=4, wxy=2, eps=1e-8
  )
  rec_hi = rdh.receiver_recover_volume(b_hi, key_e=key_e)
  assert np.array_equal(rec_hi.view(np.uint32), vol_hi.view(np.uint32))
  print(
      f"[LULUS OVERFLOW 128-BIT] Uji ekstrem be={c_hi['be']}, eps=1e-8:"
      f" fallback u128 terpicu pada {c_hi['n_u128_fallback']} voxel & pemulihan"
      " 100% eksak.\n"
  )


def compare_awcp_float_vs_int(
    vol: np.ndarray, wz: int = 1, wxy: int = 2, eps: float = 1e-4, var_win: int = 2
) -> dict:
  vp, _, _ = rdh.preprocess_volume(vol)
  pred_flt = rdh.predict_volume(vp, "AWCP_V1", wz=wz, wxy=wxy, eps=eps, var_win=var_win)
  pred_int = rdh.predict_volume(vp, "AWCP_V1_INT", wz=wz, wxy=wxy, eps=eps, var_win=var_win)

  diff_mask = pred_flt != pred_int
  pct_diff = float(np.mean(diff_mask)) * 100.0
  max_abs_diff = int(np.max(np.abs(pred_flt - pred_int)))

  c_flt = rdh.capacity_only(vol, "AWCP_V1", wz=wz, wxy=wxy, eps=eps, var_win=var_win)
  c_int = rdh.capacity_only(vol, "AWCP_V1_INT", wz=wz, wxy=wxy, eps=eps, var_win=var_win)

  return dict(
      pct_voxel_diff=pct_diff,
      n_voxel_diff=int(np.sum(diff_mask)),
      max_abs_pred_diff=max_abs_diff,
      APEC_float=c_flt["APEC"],
      APEC_int=c_int["APEC"],
      delta_APEC_int_minus_float=c_int["APEC"] - c_flt["APEC"],
      ARDC_float=c_flt["ARDC"],
      ARDC_int=c_int["ARDC"],
      AAUX_float=c_flt["AAUX"],
      AAUX_int=c_int["AAUX"],
      n_u128_fallback=c_int["n_u128_fallback"],
      worst_case_bits=c_int["overflow_info"]["worst_case_bits"],
  )


def run_prostate_stage1_evaluation(
    data_dir: str = "Task05_Prostate",
    max_files: int = 5,
    component: int = 0,
    validate_full_pipeline_on_first_n: int = 1,
):
  file_list = rdh.list_valid_nifti_files(data_dir, subdir="imagesTr")
  if not file_list:
    print(f"[PERINGATAN] Tidak ditemukan berkas NIfTI di '{data_dir}/imagesTr'.")
    return None, None

  file_list = file_list[:max_files]
  print("=" * 95)
  print(
      f"TAHAP 0 & 1: EVALUASI LENGKAP PADA {len(file_list)} VOLUME"
      f" {data_dir} (comp={component})"
  )
  print("=" * 95)

  rows = []
  flt_vs_int_rows = []

  for idx, fpath in enumerate(file_list, start=1):
    fname = os.path.basename(fpath)
    vol = rdh.load_nifti_volume(fpath, component=component, canonical=False)
    print(f"[{idx}/{len(file_list)}] {fname} -> shape={vol.shape}, N={vol.size}")

    cmp_res = compare_awcp_float_vs_int(vol, wz=1, wxy=2, eps=1e-4, var_win=2)
    cmp_res["Filename"] = fname
    flt_vs_int_rows.append(cmp_res)

    for pname in STAGE1_PREDICTORS:
      try:
        res = rdh.capacity_only(
            vol, predictor_name=pname, wz=1, wxy=2, eps=1e-4, var_win=2
        )
        # Validate against full owner_embed + extract + recover on first N files
        if idx <= validate_full_pipeline_on_first_n and pname in (
            "Causal13",
            "WCP_PAPER",
            "WCP_FAIR",
            "WCP_INSLICE",
            "AWCP_V1",
            "AWCP_V1_INT",
        ):
          full_res = rdh.evaluate_volume_pipeline(
              vol,
              predictor_name=pname,
              payload_fraction=0.90,
              wz=1,
              wxy=2,
              eps=1e-4,
              var_win=2,
          )
          assert full_res["cap"].CAUX == res["CAUX"]
          assert full_res["cap"].CPEC == res["CPEC"]
          print(
              f"    [VALIDASI PIPELINE PENUH OK] {pname:<14}: capacity_only =="
              f" owner_embed (APEC={res['APEC']:.4f}, BER=0,"
              f" pred_median={full_res['timings']['pred_kernel_median_s']:.3f}s,"
              f" embed={full_res['timings']['embed_s']:.2f}s,"
              f" recover={full_res['timings']['recover_s']:.2f}s)"
          )

        rows.append({
            "Filename": fname,
            "Component": component,
            "Predictor": pname,
            "status": "SUCCESS",
            "error_message": "",
            "Shape": str(res["shape"]),
            "N": res["N"],
            "be": res["be"],
            "ABEC": res["ABEC"],
            "ARDC": res["ARDC"],
            "ATEC": res["ATEC"],
            "AAUX": res["AAUX"],
            "Tree_Bits": res["tree_bits_len"],
            "Label_Bits": res["label_bits_len"],
            "APEC": res["APEC"],
            "CPEC": res["CPEC"],
            "MAE": res["MAE"],
            "RMSE": res["RMSE"],
            "P0": res["P0"],
            "P1": res["P1"],
            "P2_3": res["P2_3"],
            "P4_7": res["P4_7"],
            "P8_15": res["P8_15"],
            "P16_31": res["P16_31"],
            "P32_63": res["P32_63"],
            "P64+": res["P64_plus"],
            "tau_lt_001": res["tau_lt_001_frac"],
            "tau_gt_099": res["tau_gt_099_frac"],
            "tau_mean": res["tau_mean"],
            "wz_mean": res["wz_mean"],
            "u128_fallbacks": res["n_u128_fallback"],
            "Pred_s": res["pred_s"],
        })
      except (AssertionError, Exception) as e:
        print(f"    [GAGAL] {fname} | {pname}: {type(e).__name__}: {e}")
        fail_row = {col: np.nan for col in METRIC_COLUMNS}
        fail_row.update({
            "Filename": fname,
            "Component": component,
            "Predictor": pname,
            "status": "FAILED",
            "error_message": f"{type(e).__name__}: {e}",
            "Shape": str(vol.shape),
        })
        rows.append(fail_row)

  df = pd.DataFrame(rows, columns=METRIC_COLUMNS)
  df.to_csv("stage1_prostate_metrics.csv", index=False)

  df_cmp = pd.DataFrame(flt_vs_int_rows)
  df_cmp.to_csv("stage1_awcp_float_vs_int.csv", index=False)

  print("\n" + "=" * 95)
  print("STATUS KEBERHASILAN PER PREDIKTOR (SUKSES / TOTAL)")
  print("=" * 95)
  for pname in STAGE1_PREDICTORS:
    sub = df[df["Predictor"] == pname]
    n_ok = int((sub["status"] == "SUCCESS").sum())
    print(f"  {pname:<18}: {n_ok}/{len(sub)} sukses")

  pivot_ok = df.pivot(
      index=["Filename", "Component"], columns="Predictor", values="status"
  )
  valid_pairs = pivot_ok[(pivot_ok == "SUCCESS").all(axis=1)].index
  df_valid = df[df.set_index(["Filename", "Component"]).index.isin(valid_pairs)]

  print("\n" + "=" * 95)
  print(
      "TABEL 1: RINGKASAN KAPASITAS & GALAT PREDIKSI (RATA-RATA BERPASANGAN,"
      f" n={len(valid_pairs)})"
  )
  print("=" * 95)
  cap_cols = [
      "ABEC",
      "ARDC",
      "AAUX",
      "Tree_Bits",
      "Label_Bits",
      "ATEC",
      "APEC",
      "MAE",
      "RMSE",
      "Pred_s",
  ]
  sum_cap = df_valid.groupby("Predictor")[cap_cols].mean().reindex(STAGE1_PREDICTORS)
  print(sum_cap.to_string(float_format=lambda x: f"{x:.4f}"))

  print("\n" + "=" * 95)
  print(
      "TABEL 2: OKUPANSI INTERVAL RESIDUAL (P0..P64+) & DISTRIBUSI TAU/WZ"
      f" (n={len(valid_pairs)})"
  )
  print("=" * 95)
  occ_cols = [
      "P0",
      "P1",
      "P2_3",
      "P4_7",
      "P8_15",
      "P16_31",
      "P32_63",
      "P64+",
      "tau_lt_001",
      "tau_mean",
      "wz_mean",
  ]
  sum_occ = df_valid.groupby("Predictor")[occ_cols].mean().reindex(STAGE1_PREDICTORS)
  print(sum_occ.to_string(float_format=lambda x: f"{x:.4f}"))

  print("\n" + "=" * 95)
  print("TABEL 3: PERBANDINGAN AWCP_V1 (FLOAT64) VS AWCP_V1_INT (INTEGER EKSAK)")
  print("=" * 95)
  print(
      df_cmp[[
          "Filename",
          "pct_voxel_diff",
          "n_voxel_diff",
          "max_abs_pred_diff",
          "APEC_float",
          "APEC_int",
          "delta_APEC_int_minus_float",
          "n_u128_fallback",
          "worst_case_bits",
      ]].to_string(index=False)
  )
  print("-" * 95)
  print(
      "Rata-rata % voxel berbeda (AWCP_V1 vs AWCP_V1_INT):"
      f" {df_cmp['pct_voxel_diff'].mean():.4f}%"
  )
  print(
      "Rata-rata selisih APEC (AWCP_V1_INT - AWCP_V1)   :"
      f" {df_cmp['delta_APEC_int_minus_float'].mean():+.6f} bits/voxel"
  )

  return df, df_cmp


if __name__ == "__main__":
  run_stage1d_edge_and_security_tests()
  run_prostate_stage1_evaluation(
      data_dir="Task05_Prostate",
      max_files=5,
      component=0,
      validate_full_pipeline_on_first_n=1,
  )

Overwriting run_stage1.py


In [ ]:
# 1. Jalankan self-test sintetis + benchmark berpasangan Tahap 0 pada 5 volume Task05_Prostate
!python3 run_stage1.py

TAHAP 1.d: UJI KASUS TEPI (EDGE CASES), OVERFLOW, DAN KEAMANAN KUNCI
[LULUS 1/7] HU Negatif (delta=-1024.0, be=12): capacity_only == owner_embed & BER=0.
[LULUS 2/7] Volume D=1 (32x32x1): semua 7 prediktor identik antara capacity_only & pipeline penuh.
[LULUS 3/7] Volume Konstan 1-Simbol (n=1): tree_len=8 bit, label_len=1024 bit, BER=0.
[LULUS 4/7] Volume be=1 (ABEC=31.0): capacity_only == owner_embed & pemulihan eksak.
[LULUS 5/7] Volume N<512 (N=256): ditolak dengan ValueError sesuai syarat kelayakan.
[LULUS 6/7] Kunci K_E Salah: pemulihan volume gagal total, ekstraksi pesan tetap BER=0.
[LULUS 7/7] Kunci K_D Salah: BER pesan = 0.5010 (~0.5), pemulihan volume tetap BitExact=True.
[LULUS OVERFLOW 128-BIT] Uji ekstrem be=19, eps=1e-8: fallback u128 terpicu pada 238 voxel & pemulihan 100% eksak.

TAHAP 0 & 1: EVALUASI LENGKAP PADA 5 VOLUME Task05_Prostate (comp=0)
[1/5] prostate_00.nii.gz -> shape=(320, 320, 15), N=1536000
    [VALIDASI PIPELINE PENUH OK] Causal13      : capacity_only =

In [ ]:
%%writefile run_stage2_headroom.py
"""
Stage 2 Headroom Analyzer (H1, H2, H3)
Determines which companion contribution has genuine capacity headroom before Stage 3 implementation.

- H1 (LABEL_CTX headroom): Order-0 entropy H(label), conditional entropy H(label | 2 causal neighbors),
  actual Order-0 vs 4-Context Huffman code lengths, and 1-bit/symbol floor analysis on Prostate
  (comp=0 & comp=1) plus a BrainTumour-like zero-dominant (83.89% zero) volume.
- H2 (Local weight selection headroom): Tile-wise best capacity from wz in {0, 1, 2} (wxy=2, k=0 fair)
  for B in {16, 32, 64, 128}, plus Volume-Oracle, Slice-Oracle, and Voxel-Oracle (B=1), compared
  against WCP_FAIR, WCP_INSLICE, and AWCP_V1_INT.
- H3 (Template matching / CISA headroom): Non-causal per-voxel best offset (p, q) in [-R, R]^2
  for R in {1, 2, 4} with wz=1, wxy=2 (both shifted 3x3 window mean mu_z and shifted single voxel).
"""

import importlib
import os
import time
import numpy as np
import pandas as pd
from numba import njit
import rdh_encrypted_3d_medical_volumes as rdh

rdh = importlib.reload(rdh)


# ===========================================================================
# H1 Kernel: Context Extraction & Conditional Entropy / 4-Table Huffman
# ===========================================================================

@njit(cache=True)
def _build_h1_context_arrays(ell3d: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
  """For each voxel in kij scan order, return:
  - ctx_fine: exact pair index ell_left * 33 + ell_up in [0, 1088]
  - ctx_4bin: 4-state context in {0, 1, 2, 3} based on max(ell_left, ell_up):
      0: max_ell == 0 (both causal neighbors have d == 0)
      1: 1 <= max_ell <= 3 (|d| in [1, 7])
      2: 4 <= max_ell <= 6 (|d| in [8, 63])
      3: max_ell >= 7      (|d| >= 64, heavy tail)
  """
  W, H, D = ell3d.shape
  N = W * H * D
  ctx_fine = np.zeros(N, dtype=np.int64)
  ctx_4bin = np.zeros(N, dtype=np.int64)
  idx = 0
  for k in range(D):
    for i in range(W):
      for j in range(H):
        ell_left = ell3d[i, j - 1, k] if j - 1 >= 0 else 0
        ell_up = ell3d[i - 1, j, k] if i - 1 >= 0 else 0
        ctx_fine[idx] = ell_left * 33 + ell_up
        m = ell_left if ell_left >= ell_up else ell_up
        if m == 0:
          ctx_4bin[idx] = 0
        elif m <= 3:
          ctx_4bin[idx] = 1
        elif m <= 6:
          ctx_4bin[idx] = 2
        else:
          ctx_4bin[idx] = 3
        idx += 1
  return ctx_fine, ctx_4bin


def compute_h1_headroom(vol: np.ndarray, predictor_name: str = "AWCP_V1_INT", wz: int = 1, wxy: int = 2) -> dict:
  """Compute H1 (LABEL_CTX) entropy and Huffman length headroom on a volume."""
  vp, _, be = rdh.preprocess_volume(vol)
  N = vp.size
  pred = rdh.predict_volume(vp, predictor_name, wz=wz, wxy=wxy)
  res3d = vp - pred
  ell3d, z3d, s3d, label3d, cap3d = rdh.compute_residual_fields(res3d, rdh.B_WIDTH)

  labels_flat = rdh._flatten_kij(label3d).astype(np.int64)
  ctx_fine, ctx_4bin = _build_h1_context_arrays(ell3d)

  # 1. Order-0 Entropy H(label) and actual Order-0 Huffman length
  caux0, tree0, label_bits0, codes0, counts0 = rdh.compute_exact_aux_lengths_from_labels(labels_flat)
  nz0 = counts0[counts0 > 0].astype(np.float64)
  p0_dist = nz0 / float(N)
  H0_bits = float(-np.sum(p0_dist * np.log2(p0_dist)))
  L_huff0 = float(label_bits0) / float(N)

  # Zero-residual symbol (d == 0 -> z == 32, s == 0 -> label == 64) probability
  prob_d0 = float(counts0[64]) / float(N) if counts0.size > 64 else 0.0
  # Theoretical 1-bit floor excess on symbol d==0 when prob_d0 > 0.5:
  info_d0 = -prob_d0 * np.log2(prob_d0) if prob_d0 > 0 else 0.0
  floor_1bit_excess_d0 = max(0.0, prob_d0 * 1.0 - info_d0)

  # 2. Fine Conditional Entropy H(label | ell_left, ell_up) (1089 contexts, exact upper bound)
  # Map (ctx_fine, label) into 1D index: ctx_fine * 66 + label
  joint_fine = ctx_fine * 66 + labels_flat
  joint_counts = np.bincount(joint_fine, minlength=1089 * 66).reshape(1089, 66)
  ctx_totals = joint_counts.sum(axis=1)
  H_cond_fine = 0.0
  for c_idx in range(1089):
    tot = ctx_totals[c_idx]
    if tot > 0:
      row = joint_counts[c_idx]
      row_nz = row[row > 0].astype(np.float64)
      p_cond = row_nz / float(tot)
      H_c = -np.sum(p_cond * np.log2(p_cond))
      H_cond_fine += (float(tot) / float(N)) * float(H_c)

  # 3. 4-Context Binned Conditional Entropy H(label | C_4) and Actual 4-Tree Huffman Length
  joint_4 = ctx_4bin * 66 + labels_flat
  joint_4_counts = np.bincount(joint_4, minlength=4 * 66).reshape(4, 66)
  H_cond_4 = 0.0
  label_bits_ctx4 = 0
  trees_bits_ctx4 = 0
  for c_idx in range(4):
    row = joint_4_counts[c_idx]
    tot = int(row.sum())
    if tot > 0:
      row_nz = row[row > 0].astype(np.float64)
      p_cond = row_nz / float(tot)
      H_cond_4 += (float(tot) / float(N)) * float(-np.sum(p_cond * np.log2(p_cond)))
      _, t_len, l_len, _, _ = rdh.compute_exact_aux_lengths_from_labels(
          labels_flat[ctx_4bin == c_idx]
      )
      label_bits_ctx4 += l_len
      trees_bits_ctx4 += t_len
    else:
      trees_bits_ctx4 += 8  # Empty context placeholder leaf

  L_huff_ctx4 = float(label_bits_ctx4) / float(N)

  # 4. Run-Length of Zeros (RLZ) + Huffman estimate for zero-dominant regimes:
  # Group contiguous runs of label==64 (d==0) up to run length 64 as symbols
  is_zero_res = (labels_flat == 64)
  # Simple theoretical run-length bound: each run of K zeros costs ~ log2(64) bits instead of K bits
  return dict(
      predictor=predictor_name,
      N=N,
      P0_prob=prob_d0,
      H0_order0=H0_bits,
      L_huff0_actual=L_huff0,
      huff0_redundancy=L_huff0 - H0_bits,
      floor_1bit_excess_on_P0=floor_1bit_excess_d0,
      H_cond_4bin=H_cond_4,
      L_huff_ctx4_actual=L_huff_ctx4,
      H_cond_fine_2neigh=H_cond_fine,
      gain_H0_to_Hfine=H0_bits - H_cond_fine,
      gain_Lhuff0_to_Lhuff_ctx4_no_tree=L_huff0 - L_huff_ctx4,
      gain_Lhuff0_to_Lhuff_ctx4_with_tree=(
          (label_bits0 + tree0) - (label_bits_ctx4 + trees_bits_ctx4)
      ) / float(N),
      gain_Lhuff0_to_Hfine_ceiling=L_huff0 - H_cond_fine,
  )


# ===========================================================================
# H2 Kernel: Multi-Scale Local Weight Oracle (wz in {0, 1, 2})
# ===========================================================================

def compute_h2_headroom(vol: np.ndarray, tile_sizes=(16, 32, 64, 128)) -> dict:
  """Evaluate H2 headroom: choosing best wz in {0, 1, 2} (wxy=2, k=0 fair)
  at Volume, Slice, Tile BxB (B in {16, 32, 64, 128}), and Voxel (B=1) granularity.
  """
  vp, _, be = rdh.preprocess_volume(vol)
  W, H, D = vp.shape
  N = vp.size
  CBEC = N * (rdh.B_WIDTH - be)

  # Compute exact predictions, residuals, and LZC capacities for wz in {0, 1, 2}
  caps = {}
  labels = {}
  ardcs = {}
  apecs = {}
  for wz_cand in (0, 1, 2):
    pname = "WCP_INSLICE" if wz_cand == 0 else "WCP_FAIR"
    pred = rdh.predict_volume(vp, pname, wz=wz_cand, wxy=2)
    res = vp - pred
    _, _, _, lab3d, cap3d = rdh.compute_residual_fields(res, rdh.B_WIDTH)
    caux, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab3d)
    csum = rdh.capacity_summary(cap3d, be, aux_bits=caux, b=rdh.B_WIDTH)
    caps[wz_cand] = cap3d
    labels[wz_cand] = lab3d
    ardcs[wz_cand] = csum.ARDC
    apecs[wz_cand] = csum.APEC

  # Also evaluate AWCP_V1_INT on the same volume
  c_awcp = rdh.capacity_only(vol, "AWCP_V1_INT", wz=1, wxy=2, compute_tau_Metrics=False)

  # Stack capacities and labels: shape (3, W, H, D) for wz in {0, 1, 2}
  cap_stack = np.stack([caps[0], caps[1], caps[2]], axis=0)
  lab_stack = np.stack([labels[0], labels[1], labels[2]], axis=0)

  # 1. Volume-level oracle (1 best static wz for the entire volume)
  vol_totals = cap_stack.sum(axis=(1, 2, 3))
  best_vol_wz = int(np.argmax(vol_totals))
  ardc_vol_oracle = ardcs[best_vol_wz]
  apec_vol_oracle = apecs[best_vol_wz]

  # 2. Slice-level oracle (best wz in {0, 1, 2} per slice k)
  ctec_slice = 0
  lab_slice = np.empty_like(labels[0])
  for k in range(D):
    s_totals = cap_stack[:, :, :, k].sum(axis=(1, 2))
    best_m = int(np.argmax(s_totals))
    ctec_slice += int(s_totals[best_m])
    lab_slice[:, :, k] = lab_stack[best_m, :, :, k]
  caux_slice, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab_slice)
  ardc_slice_oracle = float(ctec_slice - CBEC) / float(N)
  apec_slice_oracle = float(ctec_slice - caux_slice) / float(N)

  # 3. Tile BxB oracle for B in tile_sizes (without mode signaling side-info cost)
  ardc_tiles = {}
  apec_tiles = {}
  for B in tile_sizes:
    ctec_tile = 0
    lab_tile = np.empty_like(labels[0])
    for k in range(D):
      for i0 in range(0, W, B):
        i1 = min(W, i0 + B)
        for j0 in range(0, H, B):
          j1 = min(H, j0 + B)
          sub_caps = cap_stack[:, i0:i1, j0:j1, k].sum(axis=(1, 2))
          best_m = int(np.argmax(sub_caps))
          ctec_tile += int(sub_caps[best_m])
          lab_tile[i0:i1, j0:j1, k] = lab_stack[best_m, i0:i1, j0:j1, k]
    caux_tile, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab_tile)
    ardc_tiles[B] = float(ctec_tile - CBEC) / float(N)
    apec_tiles[B] = float(ctec_tile - caux_tile) / float(N)

  # 4. Voxel-level oracle (B = 1: absolute ceiling of per-voxel weight selection in {0, 1, 2})
  best_voxel_mode = np.argmax(cap_stack, axis=0)
  ctec_voxel = int(np.max(cap_stack, axis=0).sum())
  lab_voxel = np.choose(best_voxel_mode, [lab_stack[0], lab_stack[1], lab_stack[2]])
  caux_voxel, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab_voxel)
  ardc_voxel_oracle = float(ctec_voxel - CBEC) / float(N)
  apec_voxel_oracle = float(ctec_voxel - caux_voxel) / float(N)

  return dict(
      ARDC_wz0=ardcs[0],
      ARDC_wz1_fair=ardcs[1],
      ARDC_wz2_fair=ardcs[2],
      ARDC_AWCP_V1_INT=c_awcp["ARDC"],
      ARDC_Vol_Oracle=ardc_vol_oracle,
      ARDC_Slice_Oracle=ardc_slice_oracle,
      ARDC_Tile128=ardc_tiles[128],
      ARDC_Tile64=ardc_tiles[64],
      ARDC_Tile32=ardc_tiles[32],
      ARDC_Tile16=ardc_tiles[16],
      ARDC_Voxel_Oracle=ardc_voxel_oracle,
      APEC_wz0=apecs[0],
      APEC_wz1_fair=apecs[1],
      APEC_wz2_fair=apecs[2],
      APEC_AWCP_V1_INT=c_awcp["APEC"],
      APEC_Vol_Oracle=apec_vol_oracle,
      APEC_Slice_Oracle=apec_slice_oracle,
      APEC_Tile128=apec_tiles[128],
      APEC_Tile64=apec_tiles[64],
      APEC_Tile32=apec_tiles[32],
      APEC_Tile16=apec_tiles[16],
      APEC_Voxel_Oracle=apec_voxel_oracle,
  )


# ===========================================================================
# H3 Kernel: Non-Causal Template Matching Oracle (Offsets +-1, +-2, +-4)
# ===========================================================================

@njit(cache=True)
def _h3_noncausal_oracle_numba(
    vp: np.ndarray, max_range: int, use_3x3_window: bool, wz: int = 1, wxy: int = 2
) -> np.ndarray:
  """For each voxel (i, j, k), find the non-causal offset (di_off, dj_off) in
  [-max_range, max_range]^2 on slice k-1 that minimizes |vp[i,j,k] - pred(di_off, dj_off)|
  (which monotonically maximizes LZC capacity c = min(32 - bit_length(|d|) + 1, 32)).
  Returns the optimal signed residual volume res_opt (int64).
  """
  W, H, D = vp.shape
  res_opt = np.zeros((W, H, D), dtype=np.int64)
  den_w = wz + wxy

  for k in range(D):
    for i in range(W):
      for j in range(H):
        s_intra = 0
        c_intra = 0
        if j - 1 >= 0:
          s_intra += vp[i, j - 1, k]
          c_intra += 1
        if i - 1 >= 0:
          s_intra += vp[i - 1, j, k]
          c_intra += 1
          if j - 1 >= 0:
            s_intra += vp[i - 1, j - 1, k]
            c_intra += 1
          if j + 1 < H:
            s_intra += vp[i - 1, j + 1, k]
            c_intra += 1

        if k == 0:
          pred0 = (2 * s_intra + c_intra) // (2 * c_intra) if c_intra > 0 else 0
          res_opt[i, j, k] = vp[i, j, k] - pred0
          continue

        target = vp[i, j, k]
        best_abs_d = (1 << 62)
        best_d = 0

        for p in range(-max_range, max_range + 1):
          ci = i + p
          if ci < 0 or ci >= W:
            continue
          for q in range(-max_range, max_range + 1):
            cj = j + q
            if cj < 0 or cj >= H:
              continue

            if use_3x3_window:
              s_inter = 0
              c_inter = 0
              for di in range(-1, 2):
                ii = ci + di
                if 0 <= ii < W:
                  for dj in range(-1, 2):
                    jj = cj + dj
                    if 0 <= jj < H:
                      s_inter += vp[ii, jj, k - 1]
                      c_inter += 1
            else:
              s_inter = vp[ci, cj, k - 1]
              c_inter = 1

            if c_intra == 0:
              num = wz * s_inter
              den = den_w * c_inter
            else:
              num = wz * s_inter * c_intra + wxy * s_intra * c_inter
              den = den_w * c_inter * c_intra

            pred_cand = (2 * num + den) // (2 * den)
            d_cand = target - pred_cand
            abs_d = -d_cand if d_cand < 0 else d_cand
            if abs_d < best_abs_d:
              best_abs_d = abs_d
              best_d = d_cand
              if best_abs_d == 0:
                break
          if best_abs_d == 0:
            break

        res_opt[i, j, k] = best_d
  return res_opt


def compute_h3_headroom(vol: np.ndarray, ranges=(1, 2, 4)) -> dict:
  """Evaluate H3 non-causal template matching headroom for R in {1, 2, 4}."""
  vp, _, be = rdh.preprocess_volume(vol)
  N = vp.size
  out = {}

  # Baseline wz=1 (WCP_FAIR), wz=0 (WCP_INSLICE), and AWCP_V1_INT
  for name, wz_val in (("WCP_FAIR", 1), ("WCP_INSLICE", 0), ("AWCP_V1_INT", 1)):
    c = rdh.capacity_only(vol, name, wz=wz_val, wxy=2, compute_tau_Metrics=False)
    out[f"ARDC_{name}"] = c["ARDC"]
    out[f"APEC_{name}"] = c["APEC"]

  for R in ranges:
    # 3x3 shifted window mean
    res_win = _h3_noncausal_oracle_numba(vp, R, True, 1, 2)
    _, _, _, lab_win, cap_win = rdh.compute_residual_fields(res_win, rdh.B_WIDTH)
    caux_win, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab_win)
    cs_win = rdh.capacity_summary(cap_win, be, aux_bits=caux_win, b=rdh.B_WIDTH)
    out[f"ARDC_H3_Win3x3_pm{R}"] = cs_win.ARDC
    out[f"APEC_H3_Win3x3_pm{R}"] = cs_win.APEC

    # Single shifted voxel on k-1
    res_vox = _h3_noncausal_oracle_numba(vp, R, False, 1, 2)
    _, _, _, lab_vox, cap_vox = rdh.compute_residual_fields(res_vox, rdh.B_WIDTH)
    caux_vox, _, _, _, _ = rdh.compute_exact_aux_lengths_from_labels(lab_vox)
    cs_vox = rdh.capacity_summary(cap_vox, be, aux_bits=caux_vox, b=rdh.B_WIDTH)
    out[f"ARDC_H3_Voxel_pm{R}"] = cs_vox.ARDC
    out[f"APEC_H3_Voxel_pm{R}"] = cs_vox.APEC

  return out


def run_stage2_all(data_dir: str = "Task05_Prostate", max_files: int = 5):
  rdh.warmup_jit()
  # Warm up Stage 2 kernels
  dummy = np.zeros((8, 8, 3), dtype=np.int64)
  _ = _build_h1_context_arrays(dummy)
  _ = _h3_noncausal_oracle_numba(dummy, 1, True, 1, 2)

  file_list = rdh.list_valid_nifti_files(data_dir, subdir="imagesTr")[:max_files]
  if not file_list:
    print(f"[ERROR] Berkas NIfTI tidak ditemukan di {data_dir}/imagesTr")
    return

  print("=" * 95)
  print(f"TAHAP 2: ANALISIS HEADROOM (H1, H2, H3) PADA {len(file_list)} VOLUME {data_dir}")
  print("=" * 95)

  h1_rows = []
  h2_rows = []
  h3_rows = []

  for idx, fpath in enumerate(file_list, start=1):
    fname = os.path.basename(fpath)
    t0 = time.perf_counter()
    vol = rdh.load_nifti_volume(fpath, component=0, canonical=False)

    # H1 on AWCP_V1_INT and WCP_INSLICE
    for pname in ("WCP_FAIR", "WCP_INSLICE", "AWCP_V1_INT"):
      r1 = compute_h1_headroom(vol, predictor_name=pname, wz=1, wxy=2)
      r1["Filename"] = fname
      r1["Component"] = 0
      h1_rows.append(r1)

    # H2
    r2 = compute_h2_headroom(vol, tile_sizes=(16, 32, 64, 128))
    r2["Filename"] = fname
    r2["Component"] = 0
    h2_rows.append(r2)

    # H3
    r3 = compute_h3_headroom(vol, ranges=(1, 2, 4))
    r3["Filename"] = fname
    r3["Component"] = 0
    h3_rows.append(r3)

    elapsed = time.perf_counter() - t0
    print(f"[{idx}/{len(file_list)}] Selesai mengukur H1, H2, H3 pada {fname} (comp=0) dalam {elapsed:.2f}s")

  df_h1 = pd.DataFrame(h1_rows)
  df_h2 = pd.DataFrame(h2_rows)
  df_h3 = pd.DataFrame(h3_rows)

  df_h1.to_csv("stage2_H1_headroom.csv", index=False)
  df_h2.to_csv("stage2_H2_headroom.csv", index=False)
  df_h3.to_csv("stage2_H3_headroom.csv", index=False)

  # --- Also test 1-bit floor on Component 1 (ADC modality of Prostate_00) & Zero-Dominant Volume ---
  vol_comp1 = rdh.load_nifti_volume(file_list[0], component=1, canonical=False)
  r1_comp1 = compute_h1_headroom(vol_comp1, "AWCP_V1_INT", wz=1, wxy=2)

  # Synthetic BrainTumour-like volume with 83.89% exact zeros (Table 3.1 of Proposal)
  rng = np.random.default_rng(77)
  vol_zdom = rng.integers(100, 1200, size=(128, 128, 16)).astype(np.float32)
  # Set 83.89% of spatial volume to zero background
  zero_mask = rng.random(size=vol_zdom.shape) < 0.8389
  # Make zero background spatially contiguous (outer region)
  vol_zdom[:108, :, :] = 0.0
  r1_zdom = compute_h1_headroom(vol_zdom, "AWCP_V1_INT", wz=1, wxy=2)

  # Reference AWCP_V1_INT gains on Prostate comp=0
  awcp_gain_vs_paper = 24.0434 - 23.7007  # +0.3427 bits/voxel
  awcp_gain_vs_fair = float(df_h2["APEC_AWCP_V1_INT"].mean() - df_h2["APEC_wz1_fair"].mean())
  awcp_ardc_gain_vs_fair = float(df_h2["ARDC_AWCP_V1_INT"].mean() - df_h2["ARDC_wz1_fair"].mean())

  print("\n" + "=" * 95)
  print("HASIL H1 (LABEL_CTX HEADROOM): ENTROPI ORDER-0 VS KONTEKS 2 TETANGGA KAUSAL (n=5 Prostate)")
  print("=" * 95)
  h1_cols = [
      "P0_prob",
      "H0_order0",
      "L_huff0_actual",
      "huff0_redundancy",
      "H_cond_4bin",
      "L_huff_ctx4_actual",
      "H_cond_fine_2neigh",
      "gain_Lhuff0_to_Lhuff_ctx4_no_tree",
      "gain_Lhuff0_to_Lhuff_ctx4_with_tree",
      "gain_Lhuff0_to_Hfine_ceiling",
  ]
  print(df_h1.groupby("predictor")[h1_cols].mean().to_string(float_format=lambda x: f"{x:.4f}"))

  print("\n--- Analisis Lantai 1 Bit/Simbol (1-Bit/Symbol Huffman Floor) ---")
  print(
      f"  Prostate comp=0 (T2, P0={df_h1[df_h1['predictor']=='AWCP_V1_INT']['P0_prob'].mean():.4f}): "
      f"Redundansi Huffman0 = {df_h1[df_h1['predictor']=='AWCP_V1_INT']['huff0_redundancy'].mean():.4f} b/v, "
      f"1-bit floor loss pada P0 = {df_h1[df_h1['predictor']=='AWCP_V1_INT']['floor_1bit_excess_on_P0'].mean():.4f} b/v"
  )
  print(
      f"  Prostate_00 comp=1 (ADC, P0={r1_comp1['P0_prob']:.4f}): "
      f"H0={r1_comp1['H0_order0']:.4f}, L_huff0={r1_comp1['L_huff0_actual']:.4f}, "
      f"Redundansi Huffman0 = {r1_comp1['huff0_redundancy']:.4f} b/v, "
      f"1-bit floor loss pada P0 = {r1_comp1['floor_1bit_excess_on_P0']:.4f} b/v, "
      f"Gain 4-Ctx Huffman = +{r1_comp1['gain_Lhuff0_to_Lhuff_ctx4_no_tree']:.4f} b/v"
  )
  print(
      f"  Volume Dominan-Nol (~BrainTumour, P0={r1_zdom['P0_prob']:.4f}): "
      f"H0={r1_zdom['H0_order0']:.4f}, L_huff0={r1_zdom['L_huff0_actual']:.4f}, "
      f"Redundansi Huffman0 = {r1_zdom['huff0_redundancy']:.4f} b/v, "
      f"1-bit floor loss pada P0 = {r1_zdom['floor_1bit_excess_on_P0']:.4f} b/v, "
      f"Gain 4-Ctx Huffman = +{r1_zdom['gain_Lhuff0_to_Lhuff_ctx4_no_tree']:.4f} b/v"
  )

  print("\n" + "=" * 95)
  print("HASIL H2 (PEMILIHAN BOBOT LOKAL wz in {0, 1, 2} PER TILE BxB, TANPA BIAYA SIDE-INFO)")
  print("=" * 95)
  m2 = df_h2.mean(numeric_only=True)
  h2_summary = pd.DataFrame({
      "ARDC (bits/voxel)": [
          m2["ARDC_wz1_fair"],
          m2["ARDC_wz2_fair"],
          m2["ARDC_AWCP_V1_INT"],
          m2["ARDC_wz0"],
          m2["ARDC_Vol_Oracle"],
          m2["ARDC_Slice_Oracle"],
          m2["ARDC_Tile128"],
          m2["ARDC_Tile64"],
          m2["ARDC_Tile32"],
          m2["ARDC_Tile16"],
          m2["ARDC_Voxel_Oracle"],
      ],
      "APEC (bits/voxel)": [
          m2["APEC_wz1_fair"],
          m2["APEC_wz2_fair"],
          m2["APEC_AWCP_V1_INT"],
          m2["APEC_wz0"],
          m2["APEC_Vol_Oracle"],
          m2["APEC_Slice_Oracle"],
          m2["APEC_Tile128"],
          m2["APEC_Tile64"],
          m2["APEC_Tile32"],
          m2["APEC_Tile16"],
          m2["APEC_Voxel_Oracle"],
      ],
  }, index=[
      "WCP_FAIR (wz=1, statis)",
      "WCP_FAIR (wz=2, statis)",
      "AWCP_V1_INT (baseline usulan)",
      "WCP_INSLICE (wz=0, statis)",
      "H2 Volume-Oracle (wz in {0,1,2})",
      "H2 Slice-Oracle (wz in {0,1,2})",
      "H2 Tile 128x128 (wz in {0,1,2})",
      "H2 Tile 64x64 (wz in {0,1,2})",
      "H2 Tile 32x32 (wz in {0,1,2})",
      "H2 Tile 16x16 (wz in {0,1,2})",
      "H2 Voxel-Oracle B=1 (batas mutlak)",
  ])
  h2_summary["Delta_APEC vs WCP_FAIR"] = h2_summary["APEC (bits/voxel)"] - m2["APEC_wz1_fair"]
  h2_summary["Delta_APEC vs AWCP_V1_INT"] = h2_summary["APEC (bits/voxel)"] - m2["APEC_AWCP_V1_INT"]
  h2_summary["Delta_APEC vs WCP_INSLICE"] = h2_summary["APEC (bits/voxel)"] - m2["APEC_wz0"]
  h2_summary["Rasio vs Gain AWCP_FAIR"] = h2_summary["Delta_APEC vs AWCP_V1_INT"] / awcp_gain_vs_fair
  print(h2_summary.to_string(float_format=lambda x: f"{x:.4f}"))

  print("\n" + "=" * 95)
  print("HASIL H3 (TEMPLATE MATCHING NON-KAUSAL wz=1 PADA RENTANG +-1, +-2, +-4)")
  print("=" * 95)
  m3 = df_h3.mean(numeric_only=True)
  h3_summary = pd.DataFrame({
      "ARDC (bits/voxel)": [
          m3["ARDC_WCP_FAIR"],
          m3["ARDC_AWCP_V1_INT"],
          m3["ARDC_WCP_INSLICE"],
          m3["ARDC_H3_Win3x3_pm1"],
          m3["ARDC_H3_Win3x3_pm2"],
          m3["ARDC_H3_Win3x3_pm4"],
          m3["ARDC_H3_Voxel_pm1"],
          m3["ARDC_H3_Voxel_pm2"],
          m3["ARDC_H3_Voxel_pm4"],
      ],
      "APEC (bits/voxel)": [
          m3["APEC_WCP_FAIR"],
          m3["APEC_AWCP_V1_INT"],
          m3["APEC_WCP_INSLICE"],
          m3["APEC_H3_Win3x3_pm1"],
          m3["APEC_H3_Win3x3_pm2"],
          m3["APEC_H3_Win3x3_pm4"],
          m3["APEC_H3_Voxel_pm1"],
          m3["APEC_H3_Voxel_pm2"],
          m3["APEC_H3_Voxel_pm4"],
      ],
  }, index=[
      "WCP_FAIR (wz=1, offset 0,0)",
      "AWCP_V1_INT (baseline usulan)",
      "WCP_INSLICE (wz=0)",
      "H3 Win3x3 Non-Kausal +-1 (3x3)",
      "H3 Win3x3 Non-Kausal +-2 (5x5)",
      "H3 Win3x3 Non-Kausal +-4 (9x9)",
      "H3 Single-Voxel Non-Kausal +-1 (3x3)",
      "H3 Single-Voxel Non-Kausal +-2 (5x5)",
      "H3 Single-Voxel Non-Kausal +-4 (9x9)",
  ])
  h3_summary["Delta_APEC vs WCP_FAIR"] = h3_summary["APEC (bits/voxel)"] - m3["APEC_WCP_FAIR"]
  h3_summary["Delta_APEC vs AWCP_V1_INT"] = h3_summary["APEC (bits/voxel)"] - m3["APEC_AWCP_V1_INT"]
  h3_summary["Delta_APEC vs WCP_INSLICE"] = h3_summary["APEC (bits/voxel)"] - m3["APEC_WCP_INSLICE"]
  h3_summary["Rasio vs Gain AWCP_FAIR"] = h3_summary["Delta_APEC vs AWCP_V1_INT"] / awcp_gain_vs_fair
  print(h3_summary.to_string(float_format=lambda x: f"{x:.4f}"))


if __name__ == "__main__":
  run_stage2_all(data_dir="Task05_Prostate", max_files=5)

Overwriting run_stage2_headroom.py


In [ ]:
!python3 run_stage2_headroom.py

TAHAP 2: ANALISIS HEADROOM (H1, H2, H3) PADA 5 VOLUME Task05_Prostate
[1/5] Selesai mengukur H1, H2, H3 pada prostate_00.nii.gz (comp=0) dalam 10.39s
[2/5] Selesai mengukur H1, H2, H3 pada prostate_01.nii.gz (comp=0) dalam 14.44s
[3/5] Selesai mengukur H1, H2, H3 pada prostate_02.nii.gz (comp=0) dalam 17.90s
[4/5] Selesai mengukur H1, H2, H3 pada prostate_04.nii.gz (comp=0) dalam 9.56s
[5/5] Selesai mengukur H1, H2, H3 pada prostate_06.nii.gz (comp=0) dalam 14.03s

HASIL H1 (LABEL_CTX HEADROOM): ENTROPI ORDER-0 VS KONTEKS 2 TETANGGA KAUSAL (n=5 Prostate)
             P0_prob  H0_order0  L_huff0_actual  huff0_redundancy  H_cond_4bin  L_huff_ctx4_actual  H_cond_fine_2neigh  gain_Lhuff0_to_Lhuff_ctx4_no_tree  gain_Lhuff0_to_Lhuff_ctx4_with_tree  gain_Lhuff0_to_Hfine_ceiling
predictor                                                                                                                                                                                                                 